# MiniMax H3 — open ComfyUI through Cloudflare

Select **GPU T4 ×2** and **Internet ON**, then press the single code cell's **▶ button once**. It installs the pinned runtime, verifies existing or downloaded models, starts ComfyUI and prints **Your ComfyUI Cloudflare link**. Click **Open ComfyUI in browser**.

This notebook does not queue a render, embed video previews, or copy video downloads. Work in ComfyUI through the link. Keep the Kaggle session running; the temporary public link expires when its tunnel or session stops.

Existing weights in the same session are reused. The first download is roughly 41.38 GiB and uses network, disk and CPU. Logs are saved as `comfyui-ui.log` and `cloudflared.log` under `/kaggle/working/minimax_h3_dual_t4`. The link is also saved as `comfyui-cloudflare-url.txt`.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, time, hashlib, urllib.request, urllib.error, uuid

# Edit these settings, then press this cell's Play button once. Kaggle: T4 x2, Internet ON.
PROMPT = ('A woman in a green dress dances under warm courtyard string lights. '
          'Her dress and curls swirl as the camera gently tracks her. '
          'Audio: soft acoustic guitar, footsteps and a light evening breeze.')
MODE = 'text'                       # 'text' or 'image'
IMAGE_PATH = ''                     # e.g. /kaggle/input/my-image/first-frame.png
SEED = 42
MEGAPIXELS = 0.4                    # default canvas: 512 x 768
ASPECT = (2, 3)
DURATION = 5.0                      # snaps to the H3 frame grid: 124 frames at 24 fps
MODEL_FAMILY = 'ref2va'             # 'ref2va' or 'fl2va'; matching Turbo LoRA selected
USE_BRIDGE = False                 # optional Semantic Bridge: requires MODEL_FAMILY='fl2va'
BRIDGE_ALPHA = 0.10
MODEL_STORAGE = 'auto'              # checks /kaggle/temp and /tmp; never assumes a quota
MODEL_DATASET = ''                  # optional; attached /kaggle/input files are auto-detected
ALLOW_MODEL_DOWNLOADS = True

# Keep this path to reuse weights/environment from the previous notebook in this session.
ROOT = Path('/kaggle/working/minimax_h3_dual_t4')
COMFY = ROOT/'ComfyUI'
ENV = ROOT/'venv'
PY = ENV/'bin/python'
PORT = 8188
API = f'http://127.0.0.1:{PORT}'
COMFY_COMMIT = '73c9bad4d21e7addbe1d13bc92eee0f1431b017d'  # v0.37.0
MGPU_COMMIT = 'e0232dc1d1321e23ef308a0e6a30c39c8e062b46'   # kaggle-ok-64
STEPS = 4
USE_KITCHEN_ATTENTION = True
VRAM_HEADROOM_GB = 2.0
TIMEOUT_MINUTES = 90
MODELS_READY = False
STACK_READY = False
ROOT.mkdir(parents=True, exist_ok=True)
assert MODEL_FAMILY in {'ref2va','fl2va'} and MODE in {'text','image'}
assert 0.2 <= MEGAPIXELS <= 0.4 and 1 <= DURATION <= 5 and STEPS == 4
assert len(ASPECT) == 2 and min(ASPECT) > 0 and PROMPT.strip()
assert 0 <= SEED < 2**64
assert not USE_BRIDGE or MODEL_FAMILY == 'fl2va', 'Semantic Bridge requires FL2VA; its author excludes Ref2VA.'
if MODE == 'image':
    assert Path(IMAGE_PATH).is_file(), 'Set IMAGE_PATH to an attached image.'
child_env = os.environ.copy()
for key in ('PYTHONHOME','PYTHONPATH','VIRTUAL_ENV','CONDA_PREFIX','UV_SYSTEM_PYTHON'):
    child_env.pop(key, None)
child_env.update(PIP_NO_CACHE_DIR='1', HF_HUB_DISABLE_XET='1', HF_HUB_DOWNLOAD_TIMEOUT='60',
                 CUDA_VISIBLE_DEVICES='0,1', MMH3_H3_FP16='1', MMH3_TP_CHECK='1',
                 MMH3_TP_CHUNKS='4', MMH3_TP_COMPILE='1',
                 PIP_CACHE_DIR=str(ROOT/'pip-cache'), HF_HOME=str(ROOT/'hf-cache'))
print('Settings loaded. Preparing ComfyUI browser access; no automatic rendering.')

# Embedded runtime below. Edit the settings above; no other cells need to be started.
_H3_PIPELINE_SOURCES = {'catalog': 'import signal\n'
            '\n'
            'def is_owned_server(pid):\n'
            '    """Only this notebook\'s exact Python, ComfyUI directory, and main.py command."""\n'
            "    proc = Path('/proc')/str(pid)\n"
            '    try:\n'
            "        argv = proc.joinpath('cmdline').read_bytes().split(b'\\0')\n"
            "        return (argv[0].decode() == str(PY) and b'main.py' in argv\n"
            "                and proc.joinpath('cwd').resolve() == COMFY.resolve()\n"
            "                and proc.joinpath('stat').read_text().split(') ',1)[1][0] != 'Z')\n"
            '    except (OSError, UnicodeError, IndexError):\n'
            '        return False\n'
            '\n'
            'def stop_owned_servers():\n'
            "    # Also handles the previous notebook's server after a kernel restart.\n"
            "    for proc in Path('/proc').iterdir():\n"
            '        if not proc.name.isdigit() or not is_owned_server(proc.name):\n'
            '            continue\n'
            '        pid = int(proc.name)\n'
            "        print('Closing previous H3 process:', pid, flush=True)\n"
            '        try:\n'
            '            os.kill(pid, signal.SIGTERM)\n'
            '            deadline = time.monotonic()+20\n'
            '            while is_owned_server(pid) and time.monotonic() < deadline:\n'
            '                time.sleep(0.2)\n'
            '            if is_owned_server(pid):\n'
            '                os.kill(pid, signal.SIGKILL)\n'
            '            deadline = time.monotonic()+10\n'
            '            while is_owned_server(pid) and time.monotonic() < deadline:\n'
            '                time.sleep(0.2)\n'
            '            if is_owned_server(pid):\n'
            "                raise RuntimeError(f'H3 process {pid} did not exit; refusing a second GPU "
            "process.')\n"
            '            try:\n'
            '                os.waitpid(pid, os.WNOHANG)\n'
            '            except ChildProcessError:\n'
            '                pass\n'
            '        except ProcessLookupError:\n'
            '            pass\n'
            '\n'
            'import base64,gzip\n'
            '\n'
            'original_canvas = '
            "gzip.decompress(base64.b64decode('H4sIAAAAAAACA+19aZPjNpLod/8KRk3si25PiY0bYG28F69st+1e9+HoKns2Yu1gQCJU4jRFakmqumtmvL99E6QOSjwkSqrqw9WxO1YJEJCZSOSFROKfXznOWRicXThnho7JUDE+UGMyGrAh1wNvpMcDo0cYES+gRpqzc9s/NbdhFiYx/AoVX0Q6y/04CYxfDIUFXn8dhfG78msqWfG17ZjB3/8FfzjOP4v/XUKBSTFi8UV+NzMWsB9M/mKqb8xV+I8SgKJ1lqzHsP8GGHmei7DinHqCKkHOV20cIeQSjyiKkBBUikXL76vRMjt2dThC0fr3CtV+MI70jQXgn3+svkrSwKTwlVp9MwVM4YsVJGdhPJvnm3D/c/XJkkZPC5RDi+/ZebVpSY0Xry5/eL7ZZElspyFq9e0fNYCTeb7f3O/DIJ+0zP36uj6zHTKeR9F67vPuCSYmvJnk9znDUOejiZ9V+eWQWWo0nKXJzKR5WLDveuazUZyWLH42Sqbju8EoSaszn90WfHGGXIpcVG14DQziWKidcZI6V//nbSu/Q+e58VsAcIpVuzG5D31GSRybUa6HkdngzyUki517Jl21SYSCOe0I83h7jBpNyv8uxt7awtirbeESn5GOzHVyneQ6+jn8YKKsazdj0rabmeq/m1VlN2Peazuj9u2cTZL3l8GtjkfGYp6nc/MwW30Hn+691xtm2DH55qAFbStiB4D46NtnF699ajtpRZ5y4sy/1dHcbNL5LDY6NVk+MB/0qCo68fojJTWCb47oW0oFm6SfzzJLKX9q8klSLEHbTGdTc6NnJSkvqvOCOZAl0TwHYvhZbma2eQVLp5wQNTnxSqfvguR9/DrJO1W9R6UrOSWIcC4k2RAORLgekwoEAyKYyN3CgfGKcBC0l3DA28IBNQiAxo25/jIP84I5zizWF86rMA5f6Q/Oj/Rs5875/Bj5L3+pIOgMHDa4Ap5xvgdT0fk1DEziPLGfy4+/Xl06f3WuzFTHeThyvkntwE9/i3+Lr82H3MkT57bop+PAKeTo+qv3YT4B0ZCHt8bR8yBMnDB2mFMwqOtcT1JjnGxmTFD8+L/nOgrzO2cWmpHJoJcevYOmJK4Ae/FbPHC+/vp6ng4T52Xy9vLrr50nkTVlPpDbpxfLwcteVzOdZjBznpvY7gznRufG/mCNXG709Nz51kq8X144syTNneGd85Ml8AcYLnsXzjJnmgBdkrGTT6qDvU/Sd+eO5P8GKMA8APli1k1K2fkARz2a4CwwYywVjPs/GDuvvnF0oGe5SQvBOTS5/QicNJ3l0DIxqQGVZun89dc/Ju8tVecZDGdnKVbqhzQByeG8msPvLpzIaKDyb6Xkda6TciF/O3PefP99MUFeXa1z583r4tutFXsSJRpWwxmDbhvYTQqrMcrnqSlWaGpNOifMM0cDSqPcSWFti0W1pHm7EkHALRE0J+lT10Lb0AAK/IIW9E51mJ87yBXOq5/rcP67M5/ZP7CLlu1bEIdj5y6ZOxOLvYXi17eXr4pZ/5aGefnVgqZR+A7QcMBegFXORiaGfW6bs/nw7wDUedk30iNTfgTRCyCfF5gX/ZJ5HGTFbO9hhd1yaa5BTgOXZrB8YXyTOU/COAPAliPfpHo2eXoBXJDp6SyCFTapD4wAWyobTUwwt19lN1PY3CFgB8y4YOHzFddfhTdT7VxNwnFuV+1Jibc4X2wp+tR1/ga8mBXU+ebN9Y8lCRv2o/3qvc4K2MbAQgC6A9p+mCxwuSr2Yh7OLLzAbM7Pd9dJCks+nIdRXoz/7S/fXTqYOk9GcwzeWYlmDgPrNHBimCbUxbLaHVT8LHhaigEdO69/ffHdi0snSEEepA5X6K+u8yZ2YCSiVnsQmC3OCh6zEDrvTBqDtoPtNy7An6VhbPkP8EhjIPiFpaspNuM4gl3yHnanM4YFd9gFRQXrXFBZEh+6LxC1TGJxnBqdAXPDsEa/g11JnR+++fcFvOAGw1/ANdMw32RKQDiMImD+GMRBsVVKKeY6l+Vf1zp757zSMRAf+DVzYlhYHTlPljiOQBoYK1cCixpsn7GVhBYoWEvYuu8MSGNsZy8aLN0TIHTq6BlseB0EzmBwm+rpYGJ0kCaALLbA2bUwQKOFqAJ8jR08ncd+uTL+zWzugl/mPCHLfZMaMPTgJykIx6mB6ZWddgSrCRJ4ZtEEegPGd8AtUwcgLFYBNIuTAbkLcsISTu+cgpl++PkX58nb6/90LkGfIxB0m3TjTmYpttjs5YYILIh6CEoZSD4NY5BmxahXVsCk86ljIxWWiDMwceziDqFnEGaWwYDipRgHmX09MS8TMKNe6uGFxSyfD40LzP3s/1dazg6wziwKVi8/Ks1HpfmoNB+V5qPSfFSaf0qlufYtR0mUFNGgvxBC1g7y8GbdgBA66wo7qIPDDkx4rsJSIK/8p6pxB4ZdIjnG3JOECb477kBRJe7ASa+4Azl93MGGm0HeL8TiWzMuFdqXGIT4l7MOZjn/2lBt8CcWF54TwbbJRnpmnH/9Fv9rMBis/h/+hB3BoCNnzPngKITgsxL2M1PIWXQQ8KVgCL70hO2AERfwh0Bq2QM+OJLaLzEmRReCiy6SlF2sEoWRKbFdCLP9MZW2ixRq2QX6Oh7I4w9l279seLsAixajWOlZCkYrIwq9luu7yq5+shCl2LXCFATKQnVvqSLbC/b+3+fWtLLqoBgUdv6bgrWcJ9N5BIooMv+XEisUrX0BcqXc7ldz4I6qWVHIiCCZwzoutFsYg8C1W8Q9xlB+XNjPYGFPKszlocKcMASEQox5BElYmfVIRRBZuhhhKSmVSnhsjyAyqwaR+x0Y03sIIltTxXlZHNh8oVHkNYblZvhGgwEM7nFppcF+GDiTPJ9lF8+eTeY3N6DVxuAqgInxrDDrBm/Sm2cLk33wI32Wg+X2bKrD2PoiYNfANvngT6ifmjG51UCgeWwCH4wrZQlxmya5m+mxAe8yS1JwI/6HWAPwqTP4f0vT+FkJybMgHI/nljj+4ovfYuA/J5nZHWM3B9iakXUJW2d9L7Tamg0Eyg/fnJfGdGHMwmewfCOdLv3ypxaR/wb7mt5GPiVDvzJ8fDueMV+//++mUZtwsALWB3MA/kotAhskKgQJrJPxxzMsNob8a7Vf4f8s+lGy0a8+I3R7tvDYmoMH7eu77FIsL8w9mNBBMUj3Ivu57eMz6875t8jFfnEcOQ/94XgLLVjtFkKB3NLZAvDNmMwimNEK9VCnJrqzQMU3zxajDtYMOliNtomF9brsGmTgLcAMW2BS19sD0K0QyDLO0RAKaQd/s2NlZw2Wow/K0TfBX/T7kfrLbmUv/xZv86YNwTRhki1+6A/L8cHXTY1exBLGSQQs+7QZzxEovaT0WABX8Flev7leRkcW7uCFY3VJEWVZgOpXYPXXwCL3H+Gs9GQLH9tuzNTMknMHtk6qR3npmC1hL6cukqayZ67ztzCGeTLnt7Pni96XUfTbmfUdM+uKF2MskAFBkSyCJyFI07QDsMUvbFCmmL0667kD1DMLMpnpLL9zkticO/b0V6f5EtLSZSy82eJn4C/CZvn1suDpVezt3Hllpkl653wbGR3PZ+fO6zK2YG2F1+V0P/3H4kN6k9tQ4nlpRPxoIlA9V/MwN+CAhjHMDl5s4dMv/e+law5r/2LR/irMMuA+59tyCYuR3WNjrI8q5VGlPKqUR5XyqFIeVcqWSjml08pUc4Ik+NHgVv70H7ekMzGSc1cxhJXyMOKIVWKInCuXeYII8FkxUXiPxEhZcVspFS6q/GO8VyoU2seNbc5HjPTQRC3ZkEclSlLCdmYNryaf6uxd89z1lmyii7llI0CvLq9+2iNxcxdADanYh2Rpl0bQlmNfG6vS9kczKfn+kDfleB+U/70L9sVgu4EX+wO/lRTZiERXnyUy37559c2bfuhsDbsLrX7sZI+pikhPklo7sRmxzk5HYLY97mlRm+nAL+VvI1ItzUt0rq7fvnj9Qz981kOeFpMRkMgHqR+2L1FHlyMWaHPU0yIVhPbGEGh1f3jXjFN7j4NFw8aQJ8bH3IajFj3V1NatLQ5YrMUk/bDqf09ghXCDhj3yDsHvD64Ge10lOr0mO2z6PjZJt/VxuvtN83Dw7u/lLb76LQ3pjTnxtGJajQgaGx6MgrH2FBuJoZHaYxiREcJsv3scDSbxwQcWmzup5JzNSzuLpsWyNrbVLkw09NlWdY2d1vqjsXlTGjd22ZBvzT1KKVG2tcqKjaMb7pJ7Prxh1btYonpsBlwfj/6RbHCVpUP1b3Tu2P/b6GLi3FQ1OyXV1tn8kLDgkj02oF1zxibc9Vs0DZjUuaKGXJUnGjHdYoo66ttMsUGKFTtUidJ1Q4eRoxxVRlzCiKeQxwRRlFc8VU9K11OUKi49wcWxniru46h6n6SfSh/91FP5qfhz9lPJo5/66Kc++qmPfuqjn/pJ+KkbeNFVlu7WXfdHF/bRhX10YR9d2EcX9lNzYWnNhf3Z1q8y719s+Gl1/1WB+4qQUEoyTsBTpRX3VUrmYiIw87BEiuyRH0wZsCW4xB7lEklMK6Mxyl1EJVZKcCWpoKzXsSs+3J3dcFmz/j4rOkUNmgMmb1DM2PukitA08tiJUq07xFlXSna/Gk6stm1eJjrYtWewUNQlxBNKwh7BTLLtmI9QXBHpKcH2Sar3pIu58jhVuLhvVQkhUcVcKSljCFOBEKJ9Ng07OOP+/uspVUNB+9iYy7nrJtNDGVNH7pU6Y32WdxKecz+wlcDcWXxTxaSMBx6gv8sfXrSPPJ/ZJLJ1zHEfdSgP3dfSFQxLTBBHnMmqMuTMc5WHPfgaw8beoxwbR8wlFMSAUB6xG/tE25p/0tuaPW7rx219P9ua1kup/frjlV8kQ36bTIdh3F1ODZwm7AlOJGxvxj1RvQqHscspp5KBwyBXZlbH7pacuIJzQREGS4BVDWfMmLDnPozBfIxJInqZuuyjmboI7b1hi5T0PmG1y1++e/GmZVq897TguGm/KHzaZ27LJd/YHy3yYQ8/DFrCcat7xRR/vbzfCpffh5Gxn1uW3OL//YuXz19fvnp+db8BJ39sdBrbwhdZW+AJuXgjdtEsWFt39qfmX7TL1wqdximg56c6L0qrsmooJUlmAOk8zqs73f5ksaaAqhmHHwpdGMZhlQ62NIsu7vUUN0meTYhgg+lsI2Q3Cz/442nR6W5+ywiabQYqxlbkVEyTs0zfFgESHehc18JVZ3kaTv08Ke+kQPNYR5nZmC++mSUg6+tNxcglT9fHLTCYlQ7l9pJOwiAwcX3EIhIzzwrd09QERM/qAcUlYZcU9e3ZuhoUCLmb1CvBng/L2wD2B9uNyy22QGurdY8F6maPkrEWZWQ2Ad5SqOU4sItnuoiHnT0Dth6Hqxsxz0oIn+2DcWX1N2lno2NhHKwQ2vphJRy3EXfbDsp5ot4YzIuyTXaVuYtF9Z+sdR4X9Ve3CbUi5HI74Q1rdL1nNz/v2slN5s3jfn7cz4/7+RPfzx2uBK/nfNWq1zlPFvetnnbmf1EOrrxgUnHhIUkqW49jCi4Hx/ClB23K26eEu3SxwpwRrCTbiBlg1MuNEPcUJXjz87X/7ZvXr59/e/3izetme/Prk9qYRX2/b+1pS1Y8exDp0buNkt22/XpZGaTaYivXv9a3deFUlLSPoh9sGbysoTlJi03ekNVxVt4EvIIel9FsoodFXsDGrHlycxPZA1EQraPlAVBgxnoe5fdSHYbXeHldq2ZZ2rD7EQLhUgHjSIoZ2XhShCjqUiUxQx6CHrRnBiOWvUrEyHae3XqEYEM7HMfQh2QmbL0NUNW3ZRQMb3/BDwqLHZS3sP1wAdqGhWx/IT6pU6UO5v0sY2u2PteTn5cFun6eJHnytDIncsWRzxuUZU/9Qt9anHZMuPnGwcb0Z8syWPu+bkDrl3zfXv9n4bcXVbPWa9kugLDLJfzzlABBI4QnK+dfNkzneogybPtguo8AUtxVGGEuCK8+h9LvvJl+vCAcOUVEaJHpEPinOXiW9EgRoecfFiJiXUSkvOnuA8P4xTV0f2HatsSM6G7JsYP1GgNWiyKkaf7Br0WsPpNAU0XYFKtuywssdnK4Ee2sJpn88vL67WXzIytnajAEyfH2h28OCf2nRSqYv2Syboiqvd2ia7XuJ7QvirLYkbYhPgMLy8Q3hX+Ctw8tyqIrYeG41vHpkmmobk+xEVhGY4oHZhToARtaZ8t4ajDWeqjxcCwwHXdYWJJIF0mqGIX/tRc5KgJOKjC+KGeYUGhGZI98AXXESy7keLk2DtMs9wvPrE8MvEPkVdJadllExWt7J5y7V9T/4a95kB6k+Rg3OQhBPfK0lyGB8+YjFRAoPm6G//uXby57YrAc77S55qtSTG1ImEIktuFxQHJ5Zcj7TjBfTvnri++ev2k5Sqo39bjsfNJUA7Tt2FQspz5OVsPJ5I5DyyZgtj0+ST6WU7UIuj7/AHponppNv/jjJmUv92RL44rR+yRC37v/dmkLt8c3zvtkquOyvlKUwN+BTt85o3ka3TkTHaZFxXhnHOb24QUzNamOAucmNSZ2MhB7sRPAWmROkQyRlZ1HOg4iA+aNM0rmaX5nXyuYx7bulI0dwRRFQbaiOn1sP7nOla2QNQvj7NyxhaXKIW3tewtH5mTvw9Q+KODo1D4LYfsUjyEACHliX2+I9PxmUv4iNWW9/6KCfzrNHBAPrvM3+OhoACez5b5u5mGuU7DcMvhDlwOWY+QmdavGI2WV4K0UlZrxVSdMco9JyikGJ62aHHzWu25gleH71Oo7a560vRJfyw/aS/JVf7BnkbZDrOzy+ZBCSD3y507+XAmsTUZdC6sNlq3IqQr3nsVJmBk/M8VStPMy8G3urxJWjmHsURTOViMdyuVLsboFzX4cvyGT+/N/EBaPkgC6i8qIyyOvHpuiy1UT+IiMMIGRi8H1EooJT0pvIx1MuVJKhiThiFPyMdPB+EeLREn6UdLBJHlMB/vY6WD/9ftjMthjMthj8shj8shjMthjMtjjfn7cz19GMthXCztmK4y3tKcIWXmBeL1vVvjgdS7F2ggvDeivKibScrRKhLJyqLRyWSr3WfYbDTeMRhpGWztFpZXdONoavcrt8SZ494JtTZhKOTXUNUPnaGtMMW8YjdSwLw44msciDWPhhrHoHmPR9VisC679sGTrX8qG0VS/0Xg3zVQfmolumqn9aVbJjWqEq4GT28eqcFkTXOux9lhLRLr3Zj/qS9K5N9dxir325vriaRMcldGaYVsJuZsim3Ul5TaDJ7z+qtovmXHKl5rtM5mVZEQbrCuP0jdTXz0XYcU59QRV1bxBJilyiUcUiEAhqKykGBFPuEwxWs3yoQ3JOev0R8bWOqwSKWkPC4k6ZpvvT+/ETBIO4GMpsJJIMF4NZzOpXImkAqGGJVWVcx/7yKFL7Fk+k8Je5K9QxEMwosLCw8qm9Xr0UIRXixuA4RgX6alrM+hs+bZzNUhQvZUZ9M5m2PB68UbyRWnRbJ2TZnmRQ/3CzqTOt9usd/9iK3S3arRvEhWNVLJa41sDw+bljxvPnM+sebmAs2pql5bSli+/PolcvhO5eGF+/aj7RspcESB4XcbfNhAuKDrAGyZPM1cVnEX4tnXE5HaCJmbI5awsL7HZQpTadcK5sFXbQSV7goqB710EG9kDq4cIRfg23B6GfcAIMDvwvdgy1zBRW1+g3bA3xDOdbQO/ZOFAGK2xxweECDFgCIvBUJpgMCQBlpoaTkdk2/Ddlb/SGRRdmIwvgvrZr5VfaOOb37dLSCVNvxoQxqjLuEQUb1PX0pdt2re/tx2HNhII0aHHuZCDMfOAQER6Ay+gauApLwgoG2ulWgnUkmNzDH3wPdBHHEOfADQUGWkJkwDbMDZiA6VH3kBSwRkK9FCOh230WRy9tdCmoXzlDuJ490AcdQxxJPIMCTw2AA1HBsBA44EOlBjIQEgypsGYC9VGnHqOVEca0g7C8NMTRqFjCKMI4kPFhwPsDYcDJsdioBhWA+WNApDYDI1w0EaYhvSswynD7oEy5BjKaIYMN1bUUCQHTCsz0GysBphqIgM2HIH0aaNMU+ZXZ+5XJ3EIEp3E6U5HO5h6R0lrb6wkBSINRmRIgYRDNPAwNgOjPDAYjBpKKduoVzmVPglv1cM3YI7dA7sdJb49LQIwp/FASDqCjWjQQGsQWEGg5UhRQbBhbQRbn8+30KuepNfNcNu3fax3qe6BYsfJdIyQlkoOwF2zFhMiILoQSLKAjAMErRjpNoqt8xBORDFKTk8e7yjJHjAkKFBiYJA2sAOxNwBBHwzA3fOCIQN7E+N2+WVOy0/0HqhzlHQnHvjynMBOGxIE1PH0QEmJByMlhBlhzxNG7KJOu4DvTR+2n4BvztU9mIJHSfjRkFLGwVTwJKhGZt1tMBcK/gJjChwszbw2CtazaE5DSMrFPTBal1z/qmGqxrSGNisjGBKB8WhgYLsP2HgEfIjAyRlKrEbAo2M1GrVRsZ653JnW3Ek5LLrle5SMdBT+A1asc/ZmAoOnrLo87yI/iW/43kcwJg68AFE0HiCmOTiKOADDDdMBDM0JYsIMTateaHQDD/UR6SEqcx9a0ZPRSmGitOU80Jhg/iPGBp4AqwNhTowgFIy1VqujnvDUmQ3VLQE9755oxfelVeNOXhyVbyecl7k/u7c3xm278/K5rbNXF3xtEgoR7FLYJBLsZwS2ee2QkSOC3IandZrpWcv2WyQQ1o9BZXcgqJ7nt5Xtx7cPa7dS/roCZXWytsiiFntlH5tmp7qp8EHDAXPzFLVOfzSMubpwRLfa/uikeLNy2ZtU25l62zqlubktz287ya6BOpR6501fM6/27e+9CNGaI9eUurd92WQ7Z2/rykln6l7L3l0wd9RMiE7G6ZvFXBlinpZvRTQ/xb6+mrx+hf1Zau8R35YPrz+DmZ4dMX0QpkVBg7tl0mid9be++b1Gta58x15Zj805jQ099rlS0wT9tpDruGSz5Ii9SNvJ5zvTtZZo906Lb8822WU41JJ+7lGxYXECxcZPrtgwetRsuzQb+6I1W01xP2qwvS/S3J8G6zv9owbbW4N1kPY+Ndi+0/bSYF6ba/ZtanRufm3IF23RYUJupBVQUXdCPeIiSpTkRDFbyOcQFbZRhmzh//JjdRh+CB3WcFlrS3q09+iKvVR1DaXyqx2KaS9Y69e+tkBt7dB4QWqPqEgVCyzEg2rMhhDezgjjzijjTr25HWv8fBRnu2z4spVAPXVd1dDPbDGi2rdxEpvTKIfWJPphmPuBmRV5/HWwiiREP5vp8vmxZihHsNij8uh5G9xeSgUj2aZVLCe+0vnk+YeZvUK+/yk9lrgm+DlokUN0SMNROHhX++mQBsJGela5zdLJYvVyTSfUOKvjsUZJVJfx3c6UZUNXd0m9In/j/MXr6/Nv3rx5+fzydYdIJ4ifRDEN9wB6eKhe6oPQxpXeB1BSTckyG8i3duhKt9lQUk0oHbRI9byUzdOl151wtjR3Blf5Hj7oQahYRujCpa19icwORsoO4aSPpvZ3Cu8v3QfUH57wc6cocfJEO1+DGn761Pmr84Q7A+dJc+u/OVgu/nMaA8CsiX9xYpD6KfnWU72f03Aa5uGt+T5KdL5/BPRk+r3BRzw6zEkfJswJa79Twx0k5PcNcNrxe0U3kfjC1GBjjLNmvvT11VY3lopN4TxZJqk+PftEpHvnrv3C/boThe3s3rnYGq2fTKVtftPLFz/3PVASLhNMUM/DWBLWEI5DxOUYcWlrJHF+qkwJTI4UteIhJG1b5uuGLOnsdKIjpfUcvaTuVqLtPUtdy35dhGprr7Jvb5uefbbhsbbNesTB0oFF4k54vpSbD3lZexsE97Pj4dk4cNoY/PHoaXOmo2i9PEGqN2y9XHOc8jtFRcMtodEB+G1YhjEbUeilcUWLwv3l9fPrngqXchdzyoWggmNKaxFa0MbSRRjZt6EpR+pUCtc7Ut+Sh9C3bXdzNoRcZ6cT6dv1HP1yOPBD6ttXb7573hl2au2wel27rUNDIcBPWZe27sPTJGmMo36VW0+oT4NwPJ5bHeOXeDw7DVgbanV7jkfN2prUcRjJT6tE2woMHwjb++Jish8sZcLx2hK35Tte6eksMum3xRt6q+fc9tKcBLmYcY8wKSkB7eg1eKriFIkjtH7/lYjjVKd8CNVZ3APuUgatHZar8/rNi6uurBGixEkOS27mYZOgroDa3mMJ6w+/vPju+duu5BByElizkmO7gO3osuL7y1c/v+wEl7LTZORk4c1Ud2YPtfdYQfvih1eXV120VSeBNdIgk/LyhbIuiHf1W8L98vL6efuhoCUyVw9pnjWWjdzArL3Hfji1BkUQb75Fgu7p/DMwhXQJ/N0479F1f+Q/m0PRfTTf52VF9QtYe203IN5evvo2Mjqez/azAygWLpcIYYalZFSo+g0IxlzBrIfNEeNCnOhoUJEjjwYf5G6fju/ySVgrHruxA7v67JP/8nWXiK0n8HzqIvbr/tJVPGTIeR76UzMFV80fNW6UtUTCW69kdgmk9o13D2Joz/r6zULrtM5c/eEppyEh71DXLBmPo0QHpR/d+MxVpdPIPnDTnA/Yz9s6mXAdMERdJIgiHsVIIY/Uo5PEc4XCkoIHRrBHPxXpKv4U0pXLL1+61s4sH6Xro3T9eNJ1+y7s+qHzvsLVky5jkgqJFOZceLTh6Mc+SYU4WLaMK3aym0/02LQ2+aeQrg+bLfFxpKvEn790/ZMK19orKG0St+k7eqLzeyC7bx9Z2ZSv580dYSVGJssKbBqFddktiKLWHqnJgXvycFoMQo+5bdRehOE7Y1NK9gxAsM0LrLKhitL9nEM87N0j9hAiv4ydZ7vD69lxgd9a4POwMGe9luAGqC3NexRNKLSP95Dap/WK8M4Oe10ybrfuZWNkmqrPtzpEo/T4coO5hO4SpJcNt74Plqae/BKkqfclSVMkPnlp+rBl4lorFezssFetg/bSAKJJmG6Xh/wMhellY92IL/d4rDXGoOMgmb5uyNxoCzJQTl2PcSk8kKeIs1o5QU4YcT2PeyBTKfaw94lEcNGDJck0FcuvZ8q09tpxAXm//NLKFH0STLdL8d+zXGvNCNrZYa+cotZrdEp8pgKsfb9+2aEJRgmjlNaf7Aw/bHP4oaGHyo65aJ0P6JCnSeTrcW5S/8bEZvHuahMkPePAbXcAflrkdlyZCJZgXzEtga+4EgjMW2v4NgSDPYZBTkvEqSIekieLBnN1ZDC4r6A+sVDqSKbbo8ueKXnt183kZyqattj0yJRo2F1ZcKJaRYsEytUCN43db6+2bdVvdBaOfmhKLm0/EEfWWFJSIEwoZqyhFr/1YBEprCmh2IkqDYkjj2zUQ5hT5UFcx0Zs7bDXBZTC4DlN5A4UQ1A8lrnjfGlXv/X1otffvbh+8eZ1/a23TfAf9EJQe1b07h77ZVa3uqLkMxWL7SLhy/U3vS7xeDWamGAe7S0hqT3W9qTHqQApyXn9qQXBXEqVJLAZhCLkIAFJpSu5IEQSzBVhnNXrCQiXICEUUkh6bPtaeG/5Sb8Q+ckeNBDWfnVgd4/9rh+0CiD1OQugtk33hV+sy26mMEkI1JjWAKnv8NMYmytKX3TPf5blpiiiyRpu85V3qS62YOrpULbdknuZpLq8zPrKbv83cXS3b/VL8CNdEMJKgVuJMaK17Hjwmjn0IPZFN08pdphXSTF2qaTwH4Uop7j2AKLyXCSoZIIBRISyY7M7vS9FGrMv7X54k48s2Gcqi5t23unuDqdmTG79fJ4OE59Z6eLfIhf7y0Sp4XjHEzKnEYAR4Nh0e/hY6EBepia+ySer9MxjhOP28/ErjvxueVe9WKFysX76j32rbxBWPFqPBcVcSCzrl4iJwq5SQoni+TvlHSYeiYu8lhcSsTw2/vYgmTnmQw58kuU6N34QjjrzFPfpu09yZuPT6VX5ebLSswWL7iw00t3rRJVGKpP0OglC6s+gShh74HTVsdFp/D5J32Xls4XtegW7ZE+1spfM+rIPiLrqVLB+NTT2bmjOTA3CzFIhOLu/ewqVLX1xCszbKnQ0HH9NYcXNHj1nOh9N/NF8GOkMNiAwfWsqb6ZvjK9zewd/wR6tJDwzsW0o3k7z9Wg0n84jvfjV0fcw2tLOrmbA2Ol8ejmbRXeLMjo/0j0tAulJlxBCiX1SjaF6+IoJqlwpCbQhZa/GHRa+Qq4Q4HJhJWsziKMP5Fpi/exz846EOFExgpvUmO4aBG09TpLcsRi+lzoX6CHV+T0vZrs6p43ZvxI9rJYfZAuZMVgI58GEtmr6YCSIhznhGJPRkBEmxkowSZXhwyEzZKg4YyLQ+1aC2EdcOfbppA8LoD9MdZqaslbYLy8GywEG65JhZ6e/tYfcuudS71P7itT7yN0D4Z0P5jTeDcnuwE+d+qmuR+9aLyKeH3qpBvWZ3t5CPKuTQvDDYQLeBYEfxtbJSqMGK2Gi08C3BTYbJmYNTw2VOn1e6uf6O0E6KmuUnOY1grg0GFou+UBbHPilmWM92TrvLWXqRZ1XzlILhB/p6RA24UUTVwKccZC89xfqmdTzlCLzwS87FSPUefbsvU6n85m/jArX58h1GIHRk8911NGrsATDbFF6TzXgOZzftNFp8UPf/q8thnTRzYTDJMkzcM1n/jhMs9wHKWRGOsvbhocFnySpn5osDCwWY2OCoR69azcNi1yS8BY4Moki2xUM2tRWFqzYfI03dMHk9LMpwGcvc8KMs0jftYJVvGi4zSOtw+p0NLEg7Umk0ljXsLzGXxhOjbt3o2MaZu/8fAKUmiRRULBMfW9v/AIwgxXIJimoxxKqZuJszFKQpUxjC0f7kLbedYXTLvHR8NPIlvovlWC7cGn9YUEVtk/3qiC62C2aMnChzMis2NmfJVE4KipZNkutno5G26tkK2X9HWzDMAJ59sKWJLtOejx/OcAUCdfjggghFFYIN9wCp9RV0hOCIOhw0Hk549ylWHCwWTiW9ar+QnKbtOR5FFOpFD2ywCFBD1WMf1cd/kPLyxeHM+rTvyTIvJPAuNAH6Y4A6I5u+8R0dz5FytGJignug1B3r9Pgg0+CD3g001ln1L29x+54+n5+7GKGXn4sPw2DBksR6+tAz/LuCpx7dT5RuL4+Vz/ynCbEAbDmky6StHY4SYCjHL0f4qd52HFS2n8dmLf3OAnqi+H74c5OI+KK49VO8dba4yS4L4bvh/uD5p+B0VU8TtUpNzv69EnvbX9lUt5TldWyEO7uUrn3UFCW3296XWOkbIe9feSBVBjn5sbeDgr86TzKQzBZweUNTDZKw5l1POA3V5Mkd/Dvzktgl4EunJVzZwRO5hRck9G5o52pCcL5dADTGSezvQvjIoOWu2Qe3zjvk6mOnQCcEvBzHbAvZmF054Sxk0+MM4K1MqmTjKH7MLVy5dzJknEOPaIwd7LcOr7O+zCfQIf3E/DFHOtjB0BK17m2A8BcqXasD2qnnCaZ/a09iA9HJTiLH2eRHd2ZzbOJnVzD4FHy3slm4OE7OoO+Bn59C4Bb0EJYh+KHFsjpPLOoZrMwtknxjo4DJ3uv7+znJWgF9tMwMq5zWQ5mMTbZuTOx+M3zMaCTpzrObKpdMQnAVLj/QM8kzsN4nswzACGdTcC3BbsDOpQwj23CSuRk8ziwD1rCb/MEGm20Bfy/wBkCfLFtyOYh0A9Ajss+lpLOTP9dT3VW/V474FUDBPbHUTg2N3PwbQGlcLo9AkAN7OHc3AE0eR6m5rzAfgxedAR0tmtRdrR0SmABbcM2wK7zXI8mzmii4xtjG98DqEZPI2iDQQoSWipNE8texd9AMDuuSdM5cCLQOV9StCTo+m1PGNaYdDyPCsCsZw1cD2ufT9JkfmMHcn+Lf4thZQGkyM/sC5/ZCKzci4J/ChKVX1ouHGvgQss44ThfLvW4iCGZWQYUB+a1394A9bK8AGrBooAVbFu9ufQL5ihXMU3skk6HoYFGByY0aWx0PinAi5PYD0JTgO4X/HbhXMbOfDaELudrHGfJzHIRUB0Y2I58a4CCuZnOVruk3EawLaMkuAOuzY0O7kqQbATSSSd3+WRariMwrk6nzlT/HYB/Z2CkCYw70mkKUqok5l3Ji7CjUkDYckQcZhO3HpNYySr/CoQDMPTomzIseYs7z3aRi+txIxCUfp68M/XYB6b1G0iyXgMZE3aSsO3CAbh4lJaP0vJRWj5Ky1NJy7oTfXG4CD3T0WyyOPhpiMHrmzgEuht/alNeCgegVbwuHNuLRjm7dP0umiTu0je6qMnebct0lADHWzD+Quh2AtnZ8GbdDK7TWb/AtdoVuL6yb39cWYbdN1qNCXGZh6RgAnFK6pn+jDLsEsWJADeXs8PyY4iUdgzJGhNkjq0sQvAXcntA8D/F7YFPq/iH2vv2QNM2O85JFfdTxbNQWf5t4UVf1CdZtBfHrkfW3ORtV03fmjH59fLXq8sfwKb8uZDMe0okTqkrsGLEnpAxLhtuOClXeYJjjpTC8qB8PQmrLjwlMUISEVKbRElXABBcUCawR3a8ir1bQpEvRUKxP0MWW00O95VQeZhHlW3gwD5w7EZwip3gPJH835yrmU6zML97etZLvK3Txn6yJtyHVdpYOdUApmqVdpRQjTQfy5GgYjSSaOxxJhiGXUbJWOExYloS4+1bDKm+x4+M2Y11lk+of5tp/wbG3OFZS34SaWlnKkoeFyeiPSA4yxZLuEgpapejXzXAeHYDTslsmwyNUraNu16Vr3lu25jWg7HlLRqfJ+YYVB622dAcMcXrF6Q4Qy5a/6sZhBLVS8Oh7mImFZt3DBPXSt2t5WevOqVtZCkK4dTre3QShhHPVYC65IQI6SHMGgizVaj0k6UMbaPMt+3VTzqpgylBTYxSL5bziVCAtVGgKD+59OZHqbFC+bYpvaeDHLD8GzVtxWfMKrJ1Ey2ykPuwiSLY5R5mwkOIStlw2YJb243aaxaUCEJrwoVK5sLPCVceRUjV7TIEpPcoRRiUlgLj7COQTLWR7GXy9tJ51WCSdO8tDwFNgFVs3RMiG168Y4QLl0iQ1piDRUxpja7WJGZEgE0spcSU16vNYSVgEoIYtzYtYp7ck3KMsf3I1qjjmsysvWpOl/HD0tIhrLkxi5KGTNGzXNuwn99Sy7psXPwUt1e67heaIR3g16qV7Q9+/aHlvcBflHjqhYHqwMC7LwRar3yVJWL61YQTXSxED0bBwwetQVnwuR8GnWuAD8ZgQFAnCq0PZwAX9UOBoK5FGNSe6NjEgXcxEsYH4fD9yzeX1/1wIPgIHGQHDuIgDMp8tn4Y0CMw8DpX4cFQYEeggDtVAnsoHCg5AgfVtQz0oVBQXfu5Xtb8ZIqhDYWy3HQvFDzvfvTCLqlKTqYYaq99b67C4dZRw7PNe63CIsesHw7dnOTdl4l3UhzUPWmGhnI4exkZ/Xc07RRK9admezATP8zK+Lon/LILfnRfZl4b9OV1gX4YqPvBYJdAwidEoUum4sMtVYwexF2jDN2TQEUPJotYl41XL6bRYyMchEJ50NIPgy4Try4R+0gi9mA4dEojfm+ef5txsax63w+JToFEj1gHeZidau/x9cPgvuTRDgxOJpA4OsKyOAYDcjqlwI/xm/ERONAT4nBM/AIfgwQ/nYnHu4WSPEIoeQc6npXbNv1QUUeggg+Xr+yEmpp7R/AUOYKl2pBYXBvthYToVNb8CJOJP5jBITrdaH6M66MeDIdON5ofYzTRh8JBonsyXbF4MBRwJwrovlDAJ5ew4hjLQxzBUK3uaG/5yo/AgB4hX8XJMDgm2M2OwECeDINObxTJe7P91IkwIN0x1qNiGvKBTq6oZPe1l+lhgY3tVWjMESiqS28l5tlSuavcgY3khDKlwNdBYAJ/eOfPi/zedXrmco7yv/b7gm724adxeLNK0t2a9WycJnFu4uDXdZFc7HLp4kXyw9mvP1755e3rWWpuQ/N+s0ZTvX3xMCGqtL8yuQ50rotrzhvVsIrmn4yZvbAXnew1uvLXlR4NFGmhxaIxqOaPnmUjXWSpIFcSW53IvuXjISXYysK0VbGyokjAOluDeAy5QiLpESyYINXdOGAe4y6WXBGPM4+QZZJ0uRh/rEi/LjyMXPbVH/8LflI0O65tAQA='))\n"
            'assert hashlib.sha256(original_canvas).hexdigest() == '
            "'34bccd59d74e667c5057f974c552fb961d4cd058c8a317e2146678add08328d5'\n"
            "(ROOT/'MiniMax-H3-Semantic-Bridge-Fastvideo-Ref2va-updated.json').write_bytes(original_canvas)\n"
            '\n'
            '\n'
            "ASSETS = [{'repo': 'Comfy-Org/MiniMax-H3', 'revision': "
            "'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': "
            "'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', 'family': 'ref2va', "
            "'size': 20970379616, 'sha256': "
            "'9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779'}, {'repo': "
            "'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': "
            "'loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors', 'family': 'ref2va', 'size': "
            "1956193000, 'sha256': '5b9ab5ade15d0775676d01a907268a69a1468dc6033b3b0d3ded5502f3ebb84c'}, "
            "{'repo': 'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', "
            "'file': 'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', 'family': 'fl2va', "
            "'size': 20970379616, 'sha256': "
            "'e889202c41dafb67b10d67b97f0d8541508036a6090af23425a5c2615d03c47a'}, {'repo': "
            "'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': "
            "'loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors', 'family': 'fl2va', "
            "'size': 1956192992, 'sha256': "
            "'c396a9a06f58399e9df9754b18299818d84a2ddd371724ba48fe4a41221437dc'}, {'repo': "
            "'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': "
            "'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'size': 15687142551, 'sha256': "
            "'35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6'}, {'repo': "
            "'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': "
            "'vae/minimax_h3_video_vae_fp16.safetensors', 'size': 5207808496, 'sha256': "
            "'7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522'}, {'repo': "
            "'Comfy-Org/MiniMax-H3', 'revision': 'e5eb578a89295337b8ff433a035929ce0279e0b6', 'file': "
            "'vae/minimax_h3_audio_vae_fp32.safetensors', 'size': 605254808, 'sha256': "
            "'8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48'}, {'repo': "
            "'speach1sdef178/MiniMax-H3-Semantic-Bridge', 'revision': "
            "'b9fe58ba6f428d990a59f20f09f719c8fbc67f7d', 'file': 'MiniMaxH3_SemanticBridge_v1.safetensors', "
            "'dest': 'semantic_bridge/MiniMaxH3_SemanticBridge_v1.safetensors', 'size': 11023032, 'sha256': "
            "'ac0dc8ac05f545ebdee12e2fcebe4515b049f9cfd9558eb4887a9bf3fd6d562e'}, {'repo': "
            "'speach1sdef178/MiniMax-H3-Semantic-Bridge', 'revision': "
            "'b9fe58ba6f428d990a59f20f09f719c8fbc67f7d', 'file': 'MiniMax_H3_Semantic_Bridge_v1.0.zip', "
            "'dest': None, 'size': 4406, 'sha256': "
            "'c4866a0cc554237046814c9cb827a5960caecbd2214d5a5a04b0e43716b437bb'}, {'repo': "
            "'barelymining/ComfyUI-MiniMax-H3-FastVideo', 'revision': "
            "'aa554050fe27ec3c62e20e57519beb46741c9000', 'file': 'fasth3_vsa_gate.safetensors', 'dest': "
            "'loras/fasth3_vsa_gate.safetensors', 'size': 3853522616, 'sha256': "
            "'bf95408e335c4b0d6a1a44946a428a3e12849e6eab2de9566ecedae4a0043420', 'vsa': True}]\n"
            '\n'
            'def selected_model_assets():\n'
            "    return [a for a in ASSETS if (not a.get('family') or a['family']==MODEL_FAMILY)\n"
            "            and not a.get('vsa') and (USE_BRIDGE or "
            "a['repo']!='speach1sdef178/MiniMax-H3-Semantic-Bridge')]\n"
            '\n'
            'def asset_target(a):\n'
            "    if a['file'].endswith('.zip'):return ROOT/'downloads'/a['file']\n"
            "    return COMFY/'models'/(a.get('dest') or a['file'])\n"
            '\n'
            'def stored_asset(root,a):\n'
            "    return root/a['repo'].replace('/','--')/a['revision']/a['file']\n"
            '\n'
            'def file_has_size(path,a):\n'
            "    return path.is_file() and path.stat().st_size==a['size']\n"
            '\n'
            'def model_storage_plan(package_reserve=0):\n'
            '    """Measure scratch volumes; never add capacities of directories on one device."""\n'
            '    assets=selected_model_assets()\n'
            "    wanted={Path(a['file']).name for a in assets}\n"
            '    inputs={name:[] for name in wanted}\n'
            "    roots=([Path(MODEL_DATASET)] if MODEL_DATASET else [])+[Path('/kaggle/input')]\n"
            '    for root in dict.fromkeys(roots):\n'
            '        if not root.is_dir():continue\n'
            '        for directory,_,files in os.walk(root):\n'
            '            for name in wanted.intersection(files):inputs[name].append(Path(directory)/name)\n'
            "    candidates=([Path(MODEL_STORAGE)] if MODEL_STORAGE!='auto' else\n"
            '                '
            "[Path('/kaggle/temp/minimax_h3_models'),Path('/tmp/minimax_h3_models'),ROOT/'model-store'])\n"
            '    working_free=shutil.disk_usage(ROOT).free\n'
            '    working_device=ROOT.stat().st_dev\n'
            "    print(f'Working storage: {working_free/2**30:.2f} GiB free')\n"
            '    plans=[]\n'
            '    for store in candidates:\n'
            '        try:\n'
            '            store.mkdir(parents=True,exist_ok=True)\n'
            '            free=shutil.disk_usage(store).free\n'
            '            device=store.stat().st_dev\n'
            '        except OSError as error:\n'
            "            print('Unavailable storage:',store,type(error).__name__);continue\n"
            '        locations=[];pending=[]\n'
            '        for a in assets:\n'
            '            dst=asset_target(a)\n'
            "            options=[dst,*inputs[Path(a['file']).name],stored_asset(store,a)]\n"
            '            source=next((p for p in options if file_has_size(p,a)),None)\n'
            '            locations.append((a,source or stored_asset(store,a)))\n'
            '            if source is None:pending.append(a)\n'
            "        need=sum(a['size'] for a in pending)\n"
            '        # ComfyUI output stays on working storage; reserve it separately on scratch.\n'
            '        reserve=(5*2**30+package_reserve) if device==working_device else 2**30\n'
            '        fits=need+reserve<=free and working_free>=5*2**30+package_reserve\n'
            '        plans.append(dict(store=store,locations=locations,pending=pending,need=need,\n'
            '                          free=free,device=device,fits=fits))\n'
            "        print(f'{store}: {free/2**30:.2f} GiB free; need {need/2**30:.2f} GiB; '\n"
            "              f'device {device}'+(' (shares working filesystem)' if device==working_device else "
            "' (separate filesystem)'))\n"
            "    usable=[p for p in plans if p['fits']]\n"
            '    if not usable:\n'
            "        report=[dict(path=str(p['store']),free_bytes=p['free'],missing_bytes=p['need'],\n"
            "                     filesystem_device=p['device']) for p in plans]\n"
            "        (ROOT/'storage-report.json').write_text(json.dumps(report,indent=2))\n"
            "        raise RuntimeError('No measured storage location can hold the missing weights and "
            "reserves. '\n"
            "            'Check the capacities printed above. If scratch is insufficient, prepare/attach "
            "model '\n"
            "            'inputs using notebooks/ and H3_KAGGLE_SETUP.md in the GitHub repository, then "
            "rerun. '\n"
            "            'Files under /kaggle/input are detected automatically; no model copying is "
            "required.')\n"
            '    # Prefer existing files; then scratch, in the declared candidate order.\n'
            "    plan=min(usable,key=lambda p:p['need'])\n"
            "    if plan['pending'] and not ALLOW_MODEL_DOWNLOADS:\n"
            "        raise RuntimeError('Downloads disabled; attach: '+', '.join(a['file'] for a in "
            "plan['pending']))\n"
            "    print('Selected model storage:',plan['store'])\n"
            '    print(f\'Model payload: {sum(a["size"] for a in assets)/2**30:.2f} GiB; \'\n'
            '          f\'download: {plan["need"]/2**30:.2f} GiB\')\n'
            '    return plan\n'
            '\n'
            'def sha256_file(path):\n'
            '    h=hashlib.sha256()\n'
            "    with open(path,'rb') as f:\n"
            "        for block in iter(lambda:f.read(8*1024**2),b''):h.update(block)\n"
            '    return h.hexdigest()\n'
            '\n'
            'def require_models_ready():\n'
            '    if not MODELS_READY:\n'
            "        raise RuntimeError('Model setup has not completed. Fix the reported setup failure and "
            "rerun this cell '\n"
            "                           'before starting ComfyUI or rendering; inspect storage-report.json "
            "if space is insufficient.')\n"
            '    for a in selected_model_assets():\n'
            '        if not file_has_size(asset_target(a),a):\n'
            "            raise RuntimeError('A verified model is missing or changed: "
            "'+str(asset_target(a))+'. Rerun model setup.')\n",
 'install': 'STACK_READY=False\n'
            'stop_owned_servers()\n'
            'MODELS_READY=False\n'
            '# Check scratch capacity before spending time on CUDA/package installation.\n'
            'package_probe=None\n'
            'if PY.is_file():\n'
            '    try:\n'
            "        package_probe=subprocess.run([str(PY),'-I','-c',\n"
            '            "from importlib.metadata import version; assert '
            'version(\'torch\')==\'2.10.0+cu130\'; "\n'
            '            "assert version(\'comfy-kitchen\')==\'0.2.35\'; assert '
            'version(\'comfy-aimdo\')==\'0.5.5\'"],\n'
            '            capture_output=True,env=child_env,timeout=30)\n'
            '    except (OSError,subprocess.TimeoutExpired):pass\n'
            'package_reserve=0 if package_probe and package_probe.returncode==0 else 6*2**30\n'
            'model_storage_plan(package_reserve=package_reserve)\n'
            '\n'
            'def run(args,**kwargs):\n'
            "    kwargs.setdefault('env',child_env)\n"
            '    return subprocess.run([str(a) for a in args],check=True,**kwargs)\n'
            '\n'
            "p=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',\n"
            "    '--format=csv,noheader,nounits'],capture_output=True,text=True)\n"
            "assert p.returncode==0,'Enable GPU T4 x2 in Kaggle.'\n"
            'rows=[r for r in p.stdout.splitlines() if r.strip()]\n'
            "assert len(rows)==2 and all('T4' in r for r in rows),'This notebook requires exactly two NVIDIA "
            "T4 GPUs.'\n"
            "assert all(int(r.split(',')[-1].strip().split('.')[0])>=580 for r in rows),'CUDA 13 requires "
            "host driver 580+; select a compatible Kaggle session.'\n"
            'print(p.stdout.strip())\n'
            "ram=int(next(l.split()[1] for l in Path('/proc/meminfo').read_text().splitlines() if "
            "l.startswith('MemTotal:')))/2**20\n"
            "assert ram>=24,f'At least 24 GiB host RAM required; detected {ram:.1f}.'\n"
            'def ensure_environment():\n'
            '    """Create a seeded Python 3.12 environment without host pip/ensurepip."""\n'
            '    import io, zipfile\n'
            "    expected_python='3.12.12'\n"
            "    uv_version='0.9.26'\n"
            "    uv_sha256='b7e89798bd3df7dcc4b2b4ac4e2fc11d6b3ff4fe7d764aa3012d664c635e2922'\n"
            "    uv_url=('https://files.pythonhosted.org/packages/38/16/'\n"
            "            'a07593a040fe6403c36f3b0a99b309f295cbfe19a1074dbadb671d5d4ef7/'\n"
            "            'uv-0.9.26-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl')\n"
            "    assert os.uname().machine=='x86_64','This pinned bootstrap requires Linux x86_64.'\n"
            '    # Notebook kernels may themselves run inside a different virtual environment.\n'
            "    for key in ('PYTHONHOME','PYTHONPATH','VIRTUAL_ENV','CONDA_PREFIX','UV_SYSTEM_PYTHON'):\n"
            '        child_env.pop(key,None)\n'
            "    child_env.update(UV_PYTHON_INSTALL_DIR=str(ROOT/'python'),\n"
            "                     UV_CACHE_DIR=str(ROOT/'uv-cache'),UV_LINK_MODE='copy',\n"
            "                     UV_PYTHON_BIN_DIR=str(ROOT/'bootstrap/bin'),\n"
            "                     UV_PYTHON_DOWNLOADS='automatic')\n"
            '    health_code=("import sys,pip; from pathlib import Path; "\n'
            '                 f"assert sys.version.split()[0]=={expected_python!r}; "\n'
            '                 f"assert Path(sys.prefix).resolve()==Path({str(ENV)!r}).resolve(); "\n'
            '                 "assert sys.prefix!=sys.base_prefix; print(sys.version.split()[0])")\n'
            '    def healthy():\n'
            '        if not PY.is_file():return False\n'
            '        try:\n'
            "            return subprocess.run([str(PY),'-I','-c',health_code],env=child_env,\n"
            '                capture_output=True,text=True,timeout=30).returncode==0\n'
            '        except (OSError,subprocess.TimeoutExpired):return False\n'
            '    if healthy():\n'
            "        print('Reusing verified Python',expected_python,'environment:',ENV)\n"
            '        return\n'
            "    assert shutil.disk_usage(ROOT).free>10*2**30,'Free at least 10 GiB for packages.'\n"
            "    uv=ROOT/'bootstrap'/f'uv-{uv_version}'\n"
            '    # Always check the wheel digest before executing its binary.\n'
            '    with urllib.request.urlopen(uv_url,timeout=120) as response:wheel=response.read()\n'
            "    assert hashlib.sha256(wheel).hexdigest()==uv_sha256,'uv download checksum mismatch.'\n"
            '    uv.parent.mkdir(parents=True,exist_ok=True)\n'
            '    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:\n'
            "        binary=archive.read(f'uv-{uv_version}.data/scripts/uv')\n"
            '    uv.write_bytes(binary);uv.chmod(0o755)\n'
            "    run([uv,'--version'])\n"
            '    # Install Python first; a failed network request leaves the old environment intact.\n'
            "    run([uv,'python','install',expected_python])\n"
            '    if ENV.exists() or ENV.is_symlink():\n'
            "        backup=ENV.with_name(ENV.name+'.incomplete-'+uuid.uuid4().hex[:8])\n"
            '        ENV.rename(backup)\n'
            "        print('Preserved incomplete or incompatible environment at',backup)\n"
            "    run([uv,'venv','--python',expected_python,'--managed-python','--seed',ENV])\n"
            "    assert healthy(),'Python/pip environment validation failed; inspect bootstrap output "
            "above.'\n"
            "    print('Verified isolated Python',expected_python,'with pip; host ensurepip is not used.')\n"
            '\n'
            'ensure_environment()\n'
            '\n'
            "run([PY,'-m','pip','install','pip==25.3'])\n"
            "run([PY,'-m','pip','install','--index-url','https://download.pytorch.org/whl/cu130',\n"
            "    'torch==2.10.0','torchvision==0.25.0','torchaudio==2.10.0'])\n"
            'probe="""import torch\n'
            'assert torch.cuda.is_available() and torch.cuda.device_count()==2\n'
            "assert torch.version.cuda.startswith('13.')\n"
            "assert 'sm_75' in torch.cuda.get_arch_list() or 'compute_75' in torch.cuda.get_arch_list()\n"
            'for i in range(2):\n'
            ' assert torch.cuda.get_device_capability(i)==(7,5)\n'
            " x=torch.ones((64,64),device=f'cuda:{i}',dtype=torch.float16)\n"
            ' assert (x@x).float().mean().item()==64\n'
            " print(i,torch.cuda.get_device_name(i),'FP16 CUDA matmul passed')\n"
            '# Explicit host-staged transfer: no NVLink/P2P assumption.\n'
            "src=torch.arange(256,device='cuda:0',dtype=torch.float32)\n"
            'host=torch.empty(256,pin_memory=True)\n'
            "host.copy_(src);dst=host.to('cuda:1')\n"
            'assert torch.equal(dst.cpu(),torch.arange(256,dtype=torch.float32))\n'
            "print('Two-GPU CUDA and host-staged transfer probes passed')\n"
            '"""\n'
            "run([PY,'-c',probe])\n"
            '\n'
            'def pinned_checkout(url,commit,target):\n'
            '    if not target.exists():\n'
            '        target.mkdir(parents=True)\n'
            "        run(['git','init',target]);run(['git','-C',target,'remote','add','origin',url])\n"
            "    head=subprocess.run(['git','-C',target,'rev-parse','HEAD'],capture_output=True,text=True)\n"
            "    if head.returncode==0:assert head.stdout.strip()==commit,'Existing checkout differs; choose "
            "a fresh ROOT.'\n"
            '    else:\n'
            "        run(['git','-C',target,'fetch','--depth','1','origin',commit])\n"
            "        run(['git','-C',target,'checkout','--detach','FETCH_HEAD'])\n"
            '\n'
            "pinned_checkout('https://github.com/comfyanonymous/ComfyUI.git',COMFY_COMMIT,COMFY)\n"
            "MGPU=COMFY/'custom_nodes/ComfyUI-MultiGPU-Kaggle'\n"
            "pinned_checkout('https://github.com/MrKK4/ComfyUI-MultiGPU-Kaggle.git',MGPU_COMMIT,MGPU)\n"
            '# Compatibility with current Comfy-Org per-layer .comfy_quant metadata.\n'
            'compat_source=\'"""Read both ComfyUI quantization layouts without rewriting checkpoint '
            'weights."""\\nimport json\\n\\n\\ndef read_h3_quant_layers(weights):\\n    '
            "metadata=weights.metadata() or {}\\n    if \\'_quantization_metadata\\' in metadata:\\n        "
            "layers=json.loads(metadata[\\'_quantization_metadata\\'])[\\'layers\\']\\n    else:\\n        "
            'layers={}\\n        for name in weights.keys():\\n            if '
            "name.endswith(\\'.comfy_quant\\'):\\n                "
            'data=weights.get_tensor(name)\\n                '
            "layers[name[:-len(\\'.comfy_quant\\')]]=json.loads(bytes(data.tolist()).decode(\\'utf-8\\'))\\n    "
            "if not layers:\\n        raise ValueError(\\'Checkpoint contains neither legacy nor per-layer "
            "quantization metadata.\\')\\n    return layers\\n\\n\\ndef "
            'validate_h3_quant_layers(weights):\\n    layers=read_h3_quant_layers(weights)\\n    for suffix '
            "in (\\'attn.qkv_proj\\',\\'attn.out_proj\\',\\'mlp.fc1\\',\\'mlp.fc2\\'):\\n        "
            "name=\\'blocks.0.\\'+suffix\\n        conf=layers.get(name,{})\\n        if "
            "conf.get(\\'format\\')!=\\'int8_tensorwise\\' or not conf.get(\\'convrot\\'):\\n            "
            "raise ValueError(f\\'H3 tensor parallel requires INT8 ConvRot: {name}: {conf}\\')\\n    return "
            "layers\\n'\n"
            "(MGPU/'h3_quant_compat.py').write_text(compat_source)\n"
            "tp_path=MGPU/'h3_tensor_parallel.py'\n"
            'tp_source=tp_path.read_text()\n'
            'old=\'layers = json.loads(f.metadata()["_quantization_metadata"])["layers"]\'\n'
            "new='layers = read_h3_quant_layers(f)'\n"
            'if old in tp_source:\n'
            "    assert tp_source.count(old)==1,'Unexpected TP loader source; refusing an ambiguous patch.'\n"
            '    tp_source=tp_source.replace(old,new)\n'
            "    tp_source=tp_source.replace('from .tp_source import open_weights',\n"
            "        'from .tp_source import open_weights\\nfrom .h3_quant_compat import "
            "read_h3_quant_layers')\n"
            '    tp_path.write_text(tp_source)\n'
            'else:\n'
            "    assert new in tp_source and 'from .h3_quant_compat import read_h3_quant_layers' in "
            "tp_source,'Unexpected TP loader revision.'\n"
            "print('Installed H3 quantization metadata compatibility reader.')\n"
            '\n'
            "constraints=ROOT/'constraints.txt'\n"
            "constraints.write_text('torch==2.10.0+cu130\\ntorchvision==0.25.0+cu130\\ntorchaudio==2.10.0+cu130\\ncomfy-kitchen==0.2.35\\ncomfy-aimdo==0.5.5\\n')\n"
            "run([PY,'-m','pip','install','-c',constraints,'-r',COMFY/'requirements.txt',\n"
            "    'huggingface-hub==0.36.2','transformers==4.57.6'])\n"
            "run([PY,'-m','pip','check'])\n"
            "(ROOT/'installed-packages.txt').write_text(subprocess.check_output([PY,'-m','pip','freeze'],text=True,env=child_env))\n"
            'kernel_probe="""import torch\n'
            'import comfy_kitchen.backends.cuda as ck\n'
            'with torch.inference_mode():\n'
            ' for i in range(2):\n'
            '  with torch.cuda.device(i):\n'
            "   x=torch.randn(256,256,device=f'cuda:{i}',dtype=torch.float16)\n"
            '   w=torch.randint(-8,8,(256,256),device=x.device,dtype=torch.int8)\n'
            '   scales=torch.full((256,),.02,device=x.device,dtype=torch.float32)\n'
            '   y=ck.int8_linear(x,w,scales,convrot=True,convrot_groupsize=256)\n'
            '   torch.cuda.synchronize(i)\n'
            '   assert y.shape==(256,256) and torch.isfinite(y).all()\n'
            "   print('Native ConvRot INT8 kernel passed on GPU',i)\n"
            '"""\n'
            "run([PY,'-c',kernel_probe])\n"
            '\n'
            'checkpoint_source = \'"""Persist sampled H3 video/audio before decoding; allow decode-only '
            'recovery."""\\nfrom pathlib import Path\\nimport re\\nimport torch\\nfrom safetensors.torch '
            'import save_file,load_file\\nimport folder_paths\\nfrom comfy.nested_tensor import '
            'NestedTensor\\n\\n\\ndef checkpoint_path(checkpoint_id):\\n    if not '
            "re.fullmatch(r\\'[A-Za-z0-9_-]{1,80}\\',checkpoint_id):\\n        raise "
            "ValueError(\\'Checkpoint ID must contain only letters, digits, underscore or hyphen.\\')\\n    "
            "root=Path(folder_paths.get_output_directory())/\\'h3_latents\\'\\n    "
            'root.mkdir(parents=True,exist_ok=True)\\n    return '
            "root/(checkpoint_id+\\'.safetensors\\')\\n\\n\\ndef pack_latents(samples):\\n    "
            "latent=samples[\\'samples\\']\\n    if not latent.is_nested:raise ValueError(\\'Expected H3 "
            "joint video/audio latents.\\')\\n    parts=latent.unbind()\\n    if len(parts)!=2 or "
            "parts[0].ndim!=5 or parts[1].ndim!=4:\\n        raise ValueError(\\'Unexpected H3 latent "
            "layout.\\')\\n    tensors={name:value.detach().to(\\'cpu\\').contiguous() for name,value in "
            "zip((\\'video\\',\\'audio\\'),parts)}\\n    if not all(torch.isfinite(t).all() for t in "
            "tensors.values()):\\n        raise ValueError(\\'Sampler produced non-finite latents; refusing "
            "to decode.\\')\\n    return tensors\\n\\n\\nclass H3CheckpointBeforeDecode:\\n    "
            '@classmethod\\n    def INPUT_TYPES(cls):\\n        return '
            "{\\'required\\':{\\'samples\\':(\\'LATENT\\',),\\'checkpoint_id\\':(\\'STRING\\',{\\'default\\':\\'clip\\'})}}\\n    "
            "RETURN_TYPES=(\\'LATENT\\',)\\n    OUTPUT_NODE=True\\n    FUNCTION=\\'save\\'\\n    "
            "CATEGORY=\\'h3/recovery\\'\\n    def save(self,samples,checkpoint_id):\\n        "
            'tensors=pack_latents(samples)\\n        destination=checkpoint_path(checkpoint_id)\\n        '
            "partial=destination.with_suffix(\\'.partial\\')\\n        "
            "save_file(tensors,str(partial),metadata={\\'format\\':\\'h3-joint-latent-v1\\'})\\n        "
            'partial.replace(destination)\\n        return '
            "({**samples,\\'samples\\':NestedTensor([tensors[\\'video\\'],tensors[\\'audio\\']])},)\\n\\n\\nclass "
            'H3LoadDecodeCheckpoint:\\n    @classmethod\\n    def INPUT_TYPES(cls):\\n        return '
            "{\\'required\\':{\\'checkpoint_id\\':(\\'STRING\\',{\\'default\\':\\'clip\\'})}}\\n    "
            '@classmethod\\n    def VALIDATE_INPUTS(cls,checkpoint_id):\\n        try:return True if '
            "checkpoint_path(checkpoint_id).is_file() else \\'Saved latent checkpoint does not "
            "exist.\\'\\n        except ValueError as error:return str(error)\\n    "
            "RETURN_TYPES=(\\'LATENT\\',)\\n    FUNCTION=\\'load\\'\\n    CATEGORY=\\'h3/recovery\\'\\n    "
            'def load(self,checkpoint_id):\\n        '
            "tensors=load_file(str(checkpoint_path(checkpoint_id)),device=\\'cpu\\')\\n        "
            "samples={\\'samples\\':NestedTensor([tensors[\\'video\\'],tensors[\\'audio\\']])}\\n        "
            'pack_latents(samples)\\n        return '
            "(samples,)\\n\\n\\nNODE_CLASS_MAPPINGS={\\'H3CheckpointBeforeDecode\\':H3CheckpointBeforeDecode,\\n                     "
            "\\'H3LoadDecodeCheckpoint\\':H3LoadDecodeCheckpoint}\\n'\n"
            "checkpoint_plugin=COMFY/'custom_nodes/H3CheckpointRecovery'\n"
            'checkpoint_plugin.mkdir(parents=True,exist_ok=True)\n'
            "(checkpoint_plugin/'__init__.py').write_text(checkpoint_source)\n"
            'STACK_READY=True\n'
            "print('Installation and both T4 kernel probes passed.')\n",
 'weights': 'def prepare_models():\n'
            '    global MODELS_READY\n'
            "    assert STACK_READY, 'The installation stage must complete successfully before rendering.'\n"
            '    try:\n'
            '        MODELS_READY = False\n'
            '        plan = model_storage_plan()\n'
            '        selected_assets = selected_model_assets()\n'
            "        pending = plan['pending']\n"
            "        for a, source in plan['locations']:\n"
            '            dst = asset_target(a)\n'
            '            dst.parent.mkdir(parents=True, exist_ok=True)\n'
            '            source.parent.mkdir(parents=True, exist_ok=True)\n'
            '            if dst == source:\n'
            '                continue\n'
            '            if dst.is_symlink():\n'
            '                if dst.resolve() == source.resolve():\n'
            '                    continue\n'
            '                if dst.exists():\n'
            "                    raise RuntimeError('Existing model link points elsewhere: ' + str(dst))\n"
            '                dst.unlink()\n'
            '            if dst.exists():\n'
            "                raise RuntimeError('Existing model file has an unexpected size: ' + str(dst))\n"
            '            dst.symlink_to(source.resolve())\n'
            '        download_env = child_env.copy()\n'
            '        if pending:\n'
            '            try:\n'
            '                from kaggle_secrets import UserSecretsClient\n'
            "                token = UserSecretsClient().get_secret('HF_TOKEN')\n"
            '                if token:\n'
            "                    download_env['HF_TOKEN'] = token\n"
            '            except Exception:\n'
            '                pass\n'
            '        verified_downloads = set()\n'
            "        for a, source in plan['locations']:\n"
            '            if a not in pending:\n'
            '                continue\n'
            "            stage = plan['store'] / '.staging' / hashlib.sha256((a['repo'] + "
            "a['file']).encode()).hexdigest()[:12]\n"
            "            script = 'from huggingface_hub import hf_hub_download; import sys; "
            "print(hf_hub_download(repo_id=sys.argv[1],revision=sys.argv[2],filename=sys.argv[3],local_dir=sys.argv[4]))'\n"
            '            started = time.monotonic()\n'
            "            print('Downloading:', Path(a['file']).name, flush=True)\n"
            "            run([PY, '-c', script, a['repo'], a['revision'], a['file'], stage], "
            'env=download_env)\n'
            "            print(f'Download finished in {(time.monotonic() - started) / 60:.1f} min; verifying "
            "SHA-256...', flush=True)\n"
            "            staged = stage / a['file']\n"
            "            assert file_has_size(staged, a) and sha256_file(staged) == a['sha256'], 'Downloaded "
            "artifact checksum mismatch.'\n"
            '            staged.replace(source)\n'
            '            verified_downloads.add(source.resolve())\n'
            '            shutil.rmtree(stage)\n'
            '        for a in selected_assets:\n'
            '            p = asset_target(a)\n'
            '            assert file_has_size(p, a) and (p.resolve() in verified_downloads or sha256_file(p) '
            "== a['sha256']), f'Invalid model file: {p}'\n"
            "            print('Verified:', p.name)\n"
            "        (ROOT / 'model-manifest.json').write_text(json.dumps(selected_assets, indent=2))\n"
            "        print('Weight hashes verified; checking H3 quantization metadata...', flush=True)\n"
            '        metadata_probe = "import sys\\nfrom safetensors import '
            'safe_open\\nsys.path.insert(0,sys.argv[1])\\nfrom h3_quant_compat import '
            "validate_h3_quant_layers\\nwith safe_open(sys.argv[2],framework='pt',device='cpu') as "
            "weights:\\n layers=validate_h3_quant_layers(weights)\\n print('H3 INT8 ConvRot metadata "
            'validated:\',len(layers),\'layers\')\\n"\n'
            '        unet = next((a for a in selected_model_assets() if '
            "a['file'].startswith('diffusion_models/')))\n"
            "        run([PY, '-c', metadata_probe, MGPU, asset_target(unet)])\n"
            '        if USE_BRIDGE:\n'
            '            import zipfile\n'
            '            archive = next((asset_target(a) for a in selected_assets if '
            "a['file'].endswith('.zip')))\n"
            "            target = COMFY / 'custom_nodes'\n"
            '            with zipfile.ZipFile(archive) as z:\n'
            '                for member in z.infolist():\n'
            '                    resolved = (target / member.filename).resolve()\n'
            "                    assert resolved.is_relative_to(target.resolve()), 'Unsafe ZIP path.'\n"
            "                    assert member.external_attr >> 16 & 61440 != 40960, 'ZIP symlink refused.'\n"
            "                folder = target / 'MiniMax_H3_Semantic_Bridge'\n"
            '                if folder.exists():\n'
            '                    for member in z.infolist():\n'
            '                        if not member.is_dir():\n'
            '                            assert (target / member.filename).read_bytes() == z.read(member), '
            "'Existing bridge differs; use a fresh ROOT.'\n"
            '                else:\n'
            '                    z.extractall(target)\n'
            "            assert (folder / '__init__.py').is_file(), 'Unexpected bridge ZIP layout.'\n"
            '        MODELS_READY = True\n'
            "        print('All weights and metadata verified; rendering is enabled.')\n"
            '    except BaseException:\n'
            '        MODELS_READY = False\n'
            '        raise\n'
            'prepare_models()\n',
 'workflow': 'import math\n'
             'def canvas_size(mp, aspect):\n'
             '    w = max(32, round(math.sqrt(mp*1_000_000*aspect[0]/aspect[1])/32)*32)\n'
             '    h = max(32, round(math.sqrt(mp*1_000_000*aspect[1]/aspect[0])/32)*32)\n'
             '    return w, h\n'
             '\n'
             'def frame_count(seconds):\n'
             '    frames = max(5, round(seconds*24))\n'
             '    return frames + (5-frames%17)%17\n'
             '\n'
             'def sampling_graph(checkpoint_id):\n'
             '    width, height = canvas_size(MEGAPIXELS, ASPECT)\n'
             "    cond = dict(clip=['2',0], vae=['3',0], prompt=PROMPT,\n"
             '                width=width, height=height, length=frame_count(DURATION))\n'
             "    cond_type = 'MiniMaxH3ImageToVideo'\n"
             '    if USE_BRIDGE:\n'
             "        cond_type = 'MiniMaxH3DistilledImageToVideo'\n"
             "        cond.update(distilled_adapter='MiniMaxH3_SemanticBridge_v1.safetensors',\n"
             "                    alpha=BRIDGE_ALPHA, magnitude_match='per_token')\n"
             '    graph = {}\n'
             "    def add(n, kind, **inputs): graph[str(n)] = {'class_type':kind, 'inputs':inputs}\n"
             "    add(1, 'UNETLoaderH3TensorParallel', "
             "unet_name=f'minimax_h3_{MODEL_FAMILY}_pruned_int8_convrot.safetensors')\n"
             "    add(2, 'CLIPLoaderMultiGPU', clip_name='qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', "
             "type='minimax', device='cuda:1')\n"
             "    add(3, 'VAELoaderMultiGPU', "
             "vae_name='minimax_h3_video_vae_fp16.safetensors',device='cuda:0')\n"
             "    add(5, 'LoraLoaderModelOnly', model=['1',0],\n"
             "        lora_name=('minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors' if "
             "MODEL_FAMILY=='ref2va' else 'minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors'), "
             'strength_model=1.0)\n'
             "    add(6, 'MiniMaxH3SigmaShift', model=['5',0], shift_video=6.0, shift_audio=3.0)\n"
             "    model=['6',0]\n"
             "    if MODE == 'image':\n"
             '        from PIL import Image, ImageOps\n'
             '        with Image.open(IMAGE_PATH) as im:\n'
             "            im = ImageOps.exif_transpose(im).convert('RGB')\n"
             '            im = ImageOps.pad(im, (width,height), method=Image.Resampling.LANCZOS, '
             'color=(0,0,0))\n'
             "            im.save(COMFY/'input/kaggle_first_frame.png')\n"
             "        add(18, 'LoadImage', image='kaggle_first_frame.png')\n"
             "        cond['first_frame'] = ['18',0]\n"
             '    add(7, cond_type, **cond)\n'
             "    add(8, 'RandomNoise', noise_seed=SEED)\n"
             "    add(9, 'BasicGuider', model=model, conditioning=['7',0])\n"
             "    add(10, 'KSamplerSelect', sampler_name='er_sde')\n"
             "    add(11, 'BasicScheduler', model=model, scheduler='sgm_uniform', steps=STEPS, denoise=1.0)\n"
             "    add(12, 'SamplerCustomAdvanced', noise=['8',0], guider=['9',0], sampler=['10',0], "
             "sigmas=['11',0], latent_image=['7',1])\n"
             "    add(19,'H3CheckpointBeforeDecode',samples=['12',0],checkpoint_id=checkpoint_id)\n"
             '    return graph\n'
             '\n'
             'def decode_graph(checkpoint_id):\n'
             '    return {\n'
             "        '19': {'class_type':'H3LoadDecodeCheckpoint', "
             "'inputs':{'checkpoint_id':checkpoint_id}},\n"
             "        '3': {'class_type':'VAELoaderMultiGPU', 'inputs':{\n"
             "            'vae_name':'minimax_h3_video_vae_fp16.safetensors','device':'cuda:0'}},\n"
             "        '4': {'class_type':'VAELoaderMultiGPU', 'inputs':{\n"
             "            'vae_name':'minimax_h3_audio_vae_fp32.safetensors','device':'cuda:1'}},\n"
             "        '13': {'class_type':'VAEDecode', 'inputs':{'samples':['19',0],'vae':['3',0]}},\n"
             "        '14': {'class_type':'VAEDecodeAudio', 'inputs':{'samples':['19',0],'vae':['4',0]}},\n"
             "        '15': {'class_type':'CreateVideo', "
             "'inputs':{'images':['13',0],'audio':['14',0],'fps':24.0,'bit_depth':8}},\n"
             "        '16': {'class_type':'SaveVideo', 'inputs':{'video':['15',0],\n"
             '            '
             "'filename_prefix':f'kaggle_h3/{checkpoint_id}','format':'mp4','format.codec':'h264'}},\n"
             '    }\n'
             '\n'
             'def prepare_job():\n'
             "    if MODE == 'image':\n"
             "        (COMFY/'input').mkdir(parents=True, exist_ok=True)\n"
             "    sample = sampling_graph('pending')\n"
             '    identity = {\n'
             "        'workflow_version':2, 'sample_graph':sample,\n"
             "        'comfy':COMFY_COMMIT, 'multigpu':MGPU_COMMIT,\n"
             "        'weights':[(a['file'], a['sha256']) for a in selected_model_assets()],\n"
             "        'image_sha256':sha256_file(COMFY/'input/kaggle_first_frame.png') if MODE == 'image' "
             'else None,\n'
             "        'runtime':{k:child_env[k] for k in "
             "('MMH3_H3_FP16','MMH3_TP_CHUNKS','MMH3_TP_COMPILE')},\n"
             "        'kitchen_attention':USE_KITCHEN_ATTENTION,\n"
             '    }\n'
             "    run_id = 'h3_'+hashlib.sha256(json.dumps(identity, "
             'sort_keys=True).encode()).hexdigest()[:24]\n'
             "    sample['19']['inputs']['checkpoint_id'] = run_id\n"
             "    directory = ROOT/'runs'/run_id\n"
             '    directory.mkdir(parents=True, exist_ok=True)\n'
             '    decode = decode_graph(run_id)\n'
             '    for name, data in '
             "[('settings',identity),('sample_graph',sample),('decode_graph',decode)]:\n"
             "        (directory/(name+'.json')).write_text(json.dumps(data, indent=2))\n"
             '    return run_id, directory, sample, decode\n'
             '\n'
             'from contextlib import contextmanager\n'
             'import socket\n'
             '\n'
             '\n'
             'def api_request(route, payload=None, timeout=30):\n'
             '    body = json.dumps(payload).encode() if payload is not None else None\n'
             '    req = urllib.request.Request(API+route, data=body,\n'
             "        headers={'Content-Type':'application/json'} if body is not None else {})\n"
             '    try:\n'
             '        with urllib.request.urlopen(req, timeout=timeout) as response:\n'
             '            return json.load(response)\n'
             '    except urllib.error.HTTPError as error:\n'
             "        raise RuntimeError(f'ComfyUI HTTP {error.code}: {error.read().decode()[:8000]}') from "
             'error\n'
             '\n'
             '\n'
             'def tail_log(path, limit=10000):\n'
             '    if not path.exists():\n'
             "        return '(no log file)'\n"
             "    with path.open('rb') as stream:\n"
             '        stream.seek(max(0, path.stat().st_size-limit))\n'
             "        return stream.read().decode(errors='replace')\n"
             '\n'
             '\n'
             'def check_node_catalog(info, graphs):\n'
             '    for graph in graphs:\n'
             '        for node_id, node in graph.items():\n'
             "            kind = node['class_type']\n"
             '            if kind not in info:\n'
             "                raise RuntimeError(f'Required node did not load: {kind}')\n"
             "            specs = {**info[kind]['input'].get('required',{}), "
             "**info[kind]['input'].get('optional',{})}\n"
             "            for key, value in node['inputs'].items():\n"
             '                if isinstance(value,list) and len(value) == 2:\n'
             "                    assert value[0] in graph, f'Dangling link: {node_id}.{key}'\n"
             '                elif key in specs:\n'
             '                    spec = specs[key]\n'
             '                    choices = spec[0] if isinstance(spec[0],list) else (\n'
             "                        spec[1].get('options') if len(spec)>1 and isinstance(spec[1],dict) "
             'else None)\n'
             '                    if choices and isinstance(choices[0],dict):\n'
             "                        choices = [option['key'] for option in choices]\n"
             '                    if choices is not None and value not in choices:\n'
             "                        raise RuntimeError(f'Unavailable {kind}.{key}: {value}')\n"
             '\n'
             '\n'
             '@contextmanager\n'
             'def comfy_process(phase, directory, graphs):\n'
             '    stop_owned_servers()\n'
             '    with socket.socket() as probe:\n'
             "        if probe.connect_ex(('127.0.0.1', PORT)) == 0:\n"
             "            raise RuntimeError(f'Port {PORT} is used by another service. Change PORT in "
             "settings.')\n"
             "    log_path = directory/(phase+'.log')\n"
             "    flags = ['--cuda-device','all','--vram-headroom',str(VRAM_HEADROOM_GB),\n"
             "             '--preview-method','none','--disable-auto-launch']\n"
             "    flags += ['--use-ck-attention'] if USE_KITCHEN_ATTENTION else "
             "['--use-pytorch-cross-attention']\n"
             "    with log_path.open('a', buffering=1) as log:\n"
             '        log.write(f\'\\n--- {phase} started {time.strftime("%Y-%m-%d %H:%M:%S '
             'UTC",time.gmtime())} ---\\n\')\n'
             "        process = subprocess.Popen([str(PY),'-u','main.py','--listen','127.0.0.1',\n"
             "            '--port',str(PORT),*flags], cwd=COMFY, env=child_env, stdout=log, "
             'stderr=subprocess.STDOUT)\n'
             '        try:\n'
             '            deadline = time.monotonic()+240\n'
             '            while time.monotonic() < deadline:\n'
             '                if process.poll() is not None:\n'
             "                    raise RuntimeError('ComfyUI startup failed:\\n'+tail_log(log_path))\n"
             '                try:\n'
             "                    api_request('/system_stats', timeout=3)\n"
             '                    break\n'
             '                except (OSError,RuntimeError):\n'
             '                    time.sleep(2)\n'
             '            else:\n'
             "                raise TimeoutError('ComfyUI startup timed out:\\n'+tail_log(log_path))\n"
             "            check_node_catalog(api_request('/object_info'), graphs)\n"
             '            # The pinned fork must activate its protected FP16 path on both T4s.\n'
             "            if 'fp16 mixed precision enabled' not in tail_log(log_path, 100000):\n"
             "                raise RuntimeError('H3 FP16 patch did not activate. See '+str(log_path))\n"
             '            yield process, log_path\n'
             '        finally:\n'
             '            if process.poll() is None:\n'
             '                process.terminate()\n'
             '                try:\n'
             '                    process.wait(timeout=20)\n'
             '                except subprocess.TimeoutExpired:\n'
             '                    process.kill()\n'
             '                    process.wait(timeout=20)\n'
             '            if process.poll() is None:\n'
             "                raise RuntimeError('ComfyUI did not exit; the next GPU stage is blocked.')\n"
             "            print(f'{phase.capitalize()} process exited; its CUDA allocations have been "
             "released.', flush=True)\n"
             '\n'
             '\n'
             'def execute_graph(graph, process, log_path, directory, phase):\n'
             '    started = time.monotonic()\n'
             "    queued = api_request('/prompt', {'prompt':graph, 'client_id':uuid.uuid4().hex})\n"
             "    if 'prompt_id' not in queued or queued.get('node_errors'):\n"
             "        raise RuntimeError('Workflow validation failed: '+json.dumps(queued)[:8000])\n"
             "    prompt_id = queued['prompt_id']\n"
             "    (directory/(phase+'-job.json')).write_text(json.dumps(queued, indent=2))\n"
             '    deadline = started+TIMEOUT_MINUTES*60\n'
             '    next_report = started\n'
             '    log_offset = log_path.stat().st_size\n'
             '    while time.monotonic() < deadline:\n'
             '        if process.poll() is not None:\n'
             "            raise RuntimeError(f'{phase} process exited unexpectedly:\\n'+tail_log(log_path))\n"
             "        history = api_request('/history/'+prompt_id)\n"
             '        entry = history.get(prompt_id)\n'
             '        if entry:\n'
             "            (directory/(phase+'-history.json')).write_text(json.dumps(entry, indent=2))\n"
             "            status = entry.get('status',{})\n"
             "            errors = [detail for kind,detail in status.get('messages',[]) if kind in "
             "('execution_error','execution_interrupted')]\n"
             "            if errors or status.get('status_str') == 'error':\n"
             '                details = []\n'
             '                for error in errors:\n'
             '                    details.append(f"{error.get(\'node_type\',\'node\')} '
             '({error.get(\'node_id\',\'?\')}): "\n'
             '                                   f"{error.get(\'exception_type\',\'interrupted\')}: '
             '{error.get(\'exception_message\',\'\')}")\n'
             "                    details.extend(error.get('traceback',[]))\n"
             "                message = '\\n'.join(details) or json.dumps(status)\n"
             "                (directory/(phase+'-error.txt')).write_text(message)\n"
             "                raise RuntimeError(message+'\\nFull log: '+str(log_path))\n"
             "            if status.get('completed'):\n"
             "                print(f'{phase.capitalize()} completed in {(time.monotonic()-started)/60:.1f} "
             "min.', flush=True)\n"
             '                return entry\n'
             '        if time.monotonic() >= next_report:\n'
             "            print(f'{phase.capitalize()}: {(time.monotonic()-started)/60:.1f} min elapsed', "
             'flush=True)\n'
             "            with log_path.open('rb') as stream:\n"
             '                stream.seek(log_offset)\n'
             '                recent = stream.read()\n'
             '                log_offset = stream.tell()\n'
             '            if recent:\n'
             "                print(recent[-1800:].decode(errors='replace').replace('\\r','\\n'), "
             'flush=True)\n'
             '            next_report = time.monotonic()+30\n'
             '        time.sleep(2)\n'
             "    raise TimeoutError(f'{phase} exceeded {TIMEOUT_MINUTES} minutes. Log: {log_path}')\n"
             '\n'
             '\n'
             'def validate_checkpoint(run_id):\n'
             "    path = COMFY/'output/h3_latents'/(run_id+'.safetensors')\n"
             '    if not path.is_file():\n'
             '        return False\n'
             "    code = '''import sys,torch\n"
             'from safetensors import safe_open\n'
             "with safe_open(sys.argv[1],framework='pt',device='cpu') as f:\n"
             " assert f.metadata().get('format')=='h3-joint-latent-v1','Wrong checkpoint format'\n"
             " assert set(f.keys())=={'video','audio'}\n"
             " v,a=f.get_tensor('video'),f.get_tensor('audio')\n"
             ' assert v.ndim==5 and v.shape[0]==1 and v.shape[1]==24\n'
             ' assert a.ndim==4 and a.shape[0]==1 and a.shape[1]==32 and a.shape[2]==2\n'
             ' assert v.shape[-2:]==(int(sys.argv[3])//16,int(sys.argv[2])//16)\n'
             " assert torch.isfinite(v).all() and torch.isfinite(a).all(),'Non-finite checkpoint'\n"
             " print('Verified saved latents:',list(v.shape),list(a.shape))\n"
             "'''\n"
             '    width, height = canvas_size(MEGAPIXELS, ASPECT)\n'
             "    run([PY,'-c',code,path,width,height])\n"
             '    return True\n'
             '\n'
             '\n'
             'def video_paths(entry):\n'
             '    paths = []\n'
             "    for output in entry.get('outputs',{}).values():\n"
             '        for value in output.values():\n'
             '            if not isinstance(value,list):\n'
             '                continue\n'
             '            for item in value:\n'
             '                if not isinstance(item,dict) or not '
             "item.get('filename','').lower().endswith('.mp4'):\n"
             '                    continue\n'
             "                if item.get('type','output') != 'output':\n"
             '                    continue\n'
             "                path = (COMFY/'output'/item.get('subfolder','')/item['filename']).resolve()\n"
             "                if not path.is_relative_to((COMFY/'output').resolve()):\n"
             "                    raise RuntimeError('Unexpected output path from ComfyUI.')\n"
             '                if path.is_file():\n'
             '                    paths.append(path)\n'
             '    return list(dict.fromkeys(paths))\n'
             '\n'
             '\n'
             'def verify_video(path, report_path):\n'
             '    # Decode the entire short clip: file existence alone is not success.\n'
             "    code = '''import sys,json,av\n"
             'with av.open(sys.argv[1]) as f:\n'
             " assert len(f.streams.video)==1 and len(f.streams.audio)>=1,'Missing video or audio stream'\n"
             ' stream=f.streams.video[0]\n'
             ' assert stream.width==int(sys.argv[3]) and stream.height==int(sys.argv[4])\n'
             ' frames=sum(1 for _ in f.decode(video=0))\n'
             " assert frames==int(sys.argv[5]),f'Wrong frame count: {frames}'\n"
             'with av.open(sys.argv[1]) as f:\n'
             ' samples=sum(frame.samples for frame in f.decode(audio=0))\n'
             " assert samples>0,'Empty audio stream'\n"
             "report={'file':sys.argv[1],'frames':frames,'width':int(sys.argv[3]),'height':int(sys.argv[4]),'audio_samples':samples,'fps':24}\n"
             "open(sys.argv[2],'w').write(json.dumps(report,indent=2))\n"
             "print('MP4 verified:',json.dumps(report))\n"
             "'''\n"
             '    width, height = canvas_size(MEGAPIXELS, ASPECT)\n'
             "    run([PY,'-c',code,path,report_path,width,height,frame_count(DURATION)])\n"
             '    return json.loads(report_path.read_text())\n'
             '\n'
             '\n'
             'def render_clip():\n'
             "    assert STACK_READY, 'The installation stage must complete successfully before rendering.'\n"
             '    require_models_ready()\n'
             '    run_id, directory, sample, decode = prepare_job()\n'
             "    print('Run:', run_id, '|', canvas_size(MEGAPIXELS, ASPECT), '|', frame_count(DURATION), "
             "'frames', flush=True)\n"
             "    report_path = directory/'result.json'\n"
             '    # Repeated Run All does not spend GPU time on an already finished identical clip.\n'
             '    if report_path.is_file():\n'
             '        previous = json.loads(report_path.read_text())\n'
             "        path = Path(previous['file'])\n"
             '        if path.is_file():\n'
             '            verify_video(path, report_path)\n'
             "            print('Reusing completed clip. Change the seed or prompt for a new render.')\n"
             '            return path\n'
             '    try:\n'
             '        if validate_checkpoint(run_id):\n'
             "            print('Saved sampling result found; going directly to decoding.', flush=True)\n"
             '        else:\n'
             "            with comfy_process('sampling', directory, [sample,decode]) as (process, "
             'log_path):\n'
             "                execute_graph(sample, process, log_path, directory, 'sampling')\n"
             '            # This executes AFTER process.wait(): the sampler cannot retain VRAM during '
             'decode.\n'
             '            if not validate_checkpoint(run_id):\n'
             "                raise RuntimeError('Sampling returned without its checkpoint; refusing to "
             "decode.')\n"
             "        print('Starting decoding in a fresh process.', flush=True)\n"
             "        with comfy_process('decoding', directory, [decode]) as (process, log_path):\n"
             "            entry = execute_graph(decode, process, log_path, directory, 'decoding')\n"
             '        paths = video_paths(entry)\n'
             '        if len(paths) != 1:\n'
             "            raise RuntimeError(f'Expected one MP4, found {len(paths)}. See "
             "decoding-history.json.')\n"
             '        verify_video(paths[0], report_path)\n'
             '        return paths[0]\n'
             '    except BaseException as error:\n'
             "        (directory/'last-error.txt').write_text(type(error).__name__+': '+str(error))\n"
             "        print('Stopped. Diagnostics:', directory, flush=True)\n"
             "        if (COMFY/'output/h3_latents'/(run_id+'.safetensors')).is_file():\n"
             "            print('Latents are saved. Rerun this cell to retry decoding without resampling.', "
             'flush=True)\n'
             '        raise\n',
 'access': '# Start persistent ComfyUI and print the Cloudflare browser link.\n'
           'import re, socket, platform, urllib.request, subprocess, time\n'
           'from IPython.display import display, HTML\n'
           '\n'
           "assert STACK_READY and PY.is_file() and COMFY.is_dir(), 'Installation must finish before opening "
           "ComfyUI.'\n"
           "assert platform.machine().lower() in ('x86_64', 'amd64'), 'This cloudflared binary requires "
           "x86_64 Linux.'\n"
           '\n'
           "TUNNEL_BIN = ROOT / 'cloudflared'\n"
           "SERVER_LOG = ROOT / 'comfyui-ui.log'\n"
           "TUNNEL_LOG = ROOT / 'cloudflared.log'\n"
           "cloudflared_sha='053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0'\n"
           'if not TUNNEL_BIN.is_file() or sha256_file(TUNNEL_BIN)!=cloudflared_sha:\n'
           '    # Official Cloudflare binary; downloaded in the running Kaggle environment.\n'
           '    url = '
           "'https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64'\n"
           "    temp = TUNNEL_BIN.with_suffix('.download')\n"
           '    try:\n'
           '        urllib.request.urlretrieve(url, temp)\n'
           "        assert sha256_file(temp)==cloudflared_sha, 'cloudflared checksum mismatch.'\n"
           '        temp.chmod(0o755)\n'
           '        temp.replace(TUNNEL_BIN)\n'
           '    finally:\n'
           '        temp.unlink(missing_ok=True)\n'
           '\n'
           "# Stop this notebook's previous tunnel before starting a new one.\n"
           "old_tunnel = globals().get('_h3_ui_tunnel')\n"
           'if old_tunnel is not None and old_tunnel.poll() is None:\n'
           '    old_tunnel.terminate()\n'
           '    try: old_tunnel.wait(timeout=10)\n'
           '    except subprocess.TimeoutExpired:\n'
           '        old_tunnel.kill();old_tunnel.wait(timeout=10)\n'
           '\n'
           '# Reuse an already-listening ComfyUI, or launch it using the same GPU flags as the renderer.\n'
           'def local_server_ready():\n'
           '    try:\n'
           "        api_request('/system_stats', timeout=3)\n"
           '        return True\n'
           '    except Exception:\n'
           '        return False\n'
           '\n'
           'if not local_server_ready():\n'
           '    with socket.socket() as probe:\n'
           "        if probe.connect_ex(('127.0.0.1', PORT)) == 0:\n"
           "            raise RuntimeError(f'Port {PORT} is occupied by another program.')\n"
           "    ui_flags = ['--cuda-device','all','--vram-headroom',str(VRAM_HEADROOM_GB),\n"
           "                '--preview-method','none','--disable-auto-launch']\n"
           "    ui_flags += ['--use-ck-attention'] if USE_KITCHEN_ATTENTION else "
           "['--use-pytorch-cross-attention']\n"
           "    with SERVER_LOG.open('w') as log:\n"
           '        _h3_ui_server = subprocess.Popen(\n'
           "            [str(PY),'-u','main.py','--listen','127.0.0.1','--port',str(PORT),*ui_flags],\n"
           '            cwd=COMFY, env=child_env, stdout=log, stderr=subprocess.STDOUT)\n'
           '    for _ in range(120):\n'
           '        if _h3_ui_server.poll() is not None:\n'
           "            raise RuntimeError('ComfyUI exited during startup:\\n'+tail_log(SERVER_LOG))\n"
           '        if local_server_ready(): break\n'
           '        time.sleep(2)\n'
           "    else: raise TimeoutError('ComfyUI did not become ready:\\n'+tail_log(SERVER_LOG))\n"
           '\n'
           "with TUNNEL_LOG.open('w') as log:\n"
           '    _h3_ui_tunnel = subprocess.Popen(\n'
           '        '
           "[str(TUNNEL_BIN),'tunnel','--no-autoupdate','--protocol','http2','--url',f'http://127.0.0.1:{PORT}'],\n"
           '        stdout=log, stderr=subprocess.STDOUT)\n'
           '\n'
           'cloudflare_url = None\n'
           'for _ in range(90):\n'
           '    if _h3_ui_tunnel.poll() is not None:\n'
           "        raise RuntimeError('Cloudflare tunnel exited:\\n'+tail_log(TUNNEL_LOG))\n"
           "    log_text = TUNNEL_LOG.read_text(errors='replace')\n"
           "    found = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com',log_text)\n"
           '    if found:\n'
           '        cloudflare_url = found.group(0)\n'
           '        break\n'
           '    time.sleep(1)\n'
           'if not cloudflare_url:\n'
           "    raise TimeoutError('No Cloudflare URL found. Check "
           "'+str(TUNNEL_LOG)+'\\n'+tail_log(TUNNEL_LOG))\n"
           "print('Your ComfyUI Cloudflare link:',cloudflare_url)\n"
           'display(HTML(\'<a href="\'+cloudflare_url+\'" target="_blank" rel="noopener noreferrer">Open '
           "ComfyUI in browser</a>'))\n"
           "print('Keep this Kaggle session running while using ComfyUI. The temporary link grants public "
           "access.')\n"
           '\n'
           "(ROOT/'comfyui-cloudflare-url.txt').write_text(cloudflare_url+'\\n')\n"
           '\n'
           'def stop_comfyui_access():\n'
           "    for name in ('_h3_ui_tunnel','_h3_ui_server'):\n"
           '        proc = globals().get(name)\n'
           '        if proc is not None and proc.poll() is None:\n'
           '            proc.terminate()\n'
           '            try: proc.wait(timeout=10)\n'
           '            except subprocess.TimeoutExpired:\n'
           '                proc.kill();proc.wait(timeout=10)\n'
           "    print('ComfyUI browser access stopped.')\n"}

def run_pipeline(stages, namespace):
    """Run every stage in order in this one cell; propagate real failures."""
    labels = {
        'catalog':'Load model catalog and process helpers',
        'install':'Install and test the T4 runtime',
        'weights':'Prepare and verify weights',
        'workflow':'Load the render workflow',
        'access':'Open ComfyUI through Cloudflare',
    }
    namespace['RESULT_PATH'] = None
    for index, (name, source) in enumerate(stages.items(), start=1):
        label = labels[name]
        print(f'\n[{index}/{len(stages)}] {label}', flush=True)
        try:
            exec(compile(source, f'<h3:{name}>', 'exec'), namespace)
        except BaseException as error:
            print(f'\nStopped during {label}: {type(error).__name__}: {error}', flush=True)
            print('Later stages were not run. Existing model files and saved latents are retained.', flush=True)
            raise
        print(f'Completed: {label}', flush=True)
    print('\nComfyUI is running. Open the Cloudflare link above.', flush=True)

run_pipeline(_H3_PIPELINE_SOURCES, globals())


## Runtime and workflow notes

Uses Python 3.12.12, PyTorch 2.10 / CUDA 13, pinned ComfyUI and the dual-T4 tensor-parallel fork. Both GPU kernel probes run before missing weights download. CUDA 13 requires driver 580 or newer. Four-step Turbo LoRA weights are included; Semantic Bridge is optional for FL2VA. FastVideo VSA kernels are not enabled for T4.

The supplied workflow is retained for reference, not automatically queued. Browser workflows must use nodes compatible with this pinned stack. Avoid the old dual-helper VAE configuration that exhausted GPU memory. The existing isolated sampling/decoding controller remains available in the embedded runtime, but is not started automatically and may restart ComfyUI when explicitly invoked.

The uploaded run successfully completed a 124-frame, 512 × 768 video with audio using that controller. The new browser-access path has local lifecycle checks; a live Cloudflare connection from your Kaggle session still needs to be established there.

To close access without ending the Kaggle session, call `stop_comfyui_access()`. No stop cell is queued automatically by Run All.